# 🗣️ Processamento de Linguagem Natural (PLN)
### Inteligência Artificial — Ciência da Computação · Aula 09

<a href="https://colab.research.google.com/github/profluiscaparroz/inteligencia-artificial/blob/main/09-processamento-linguagem-natural/09_Processamento_Linguagem_Natural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

**Objetivo desta aula:** Compreender os fundamentos do Processamento de Linguagem Natural (PLN/NLP), suas principais técnicas de representação textual, análise e implementações práticas para tarefas como classificação de textos, análise de sentimentos e extração de informações.

---

### 🎯 Objetivos de aprendizagem

Ao final desta aula, você será capaz de:

1. **Explicar** por que a linguagem natural é difícil para computadores (ambiguidade, contexto, variação);
2. **Pré-processar** textos: tokenização, normalização, *stopwords*, *stemming*, expressões regulares e subpalavras (BPE);
3. **Medir** propriedades estatísticas da língua com um corpus real (Lei de Zipf);
4. **Representar** textos como vetores (*Bag-of-Words*, n-gramas, TF-IDF) e compará-los (similaridade de cosseno);
5. **Construir** um buscador, um corretor ortográfico e classificadores de sentimento (léxico × aprendizado de máquina);
6. **Entender** a ideia central por trás dos modelos de linguagem (prever a próxima palavra) e dos *word embeddings* (contexto define significado);
7. **Avaliar** modelos com as métricas certas: precisão, revocação e F1 para classificadores, perplexidade para modelos de linguagem;
8. **Explicar** como a atenção dos Transformers torna o significado dependente do contexto e como as peças da aula se juntam em um LLM.

### 🗺️ Roteiro da aula

| Parte | Conteúdo | Destaques práticos |
|:-:|---|---|
| 0 | Preparação do ambiente | — |
| 1 | O que é PLN, história, níveis de análise e tipos de ambiguidade | ELIZA simulada e **conversa ao vivo** 💬; ELIZA × IAs generativas |
| 2 | Pré-processamento de texto | Pipeline interativo 🎛️, *stemming* RSLP × lematização, regex, Lei de Zipf com Machado de Assis, BPE 🎬, distância de edição 🎬 e corretor ortográfico |
| 3 | Representação vetorial | BoW visual, n-gramas, TF-IDF (com a conta feita à mão) e mini buscador 🎛️ |
| 4 | Análise de sentimentos e além | Léxico × Naive Bayes (com a conta feita à mão), precisão/revocação/F1, modelo de linguagem de bigramas 🎬, perplexidade e suavização, *embeddings* por coocorrência, atenção e LLMs |
| 5–10 | Bibliotecas, aplicações, comparações, resumo da aula, exercícios e referências | Perguntas para discussão |

> 💡 **Como usar:** execute as células **em ordem**. 🎬 = animação (botão ▶), 🎛️ = controle interativo. Depois de **cada resultado** há uma seção **🔎 Como interpretar**.

---

## Parte 0 — Preparação do ambiente

Execute a célula abaixo antes das demais. Algumas seções baixam o corpus de Machado de Assis do NLTK (requer internet; no Colab funciona automaticamente).

In [2]:
# =====================================================
# IMPORTAÇÕES E FUNÇÕES AUXILIARES — execute primeiro
# =====================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

# Animações renderizadas como player HTML/JavaScript (funciona no Colab sem ffmpeg)
plt.rcParams['animation.html'] = 'jshtml'
plt.rcParams['animation.embed_limit'] = 60
plt.rcParams['figure.dpi'] = 90

CORES = {'azul': '#1f77b4', 'vermelho': '#d62728', 'verde': '#2ca02c', 'laranja': '#ff7f0e',
         'roxo': '#9467bd', 'cinza': '#bbbbbb', 'amarelo': '#ffd54f'}


def mostrar_animacao(anim):
    # Exibe uma animação do matplotlib como player interativo (▶ ⏸ ⏭) e fecha a figura estática.
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


print('✅ Ambiente pronto!')

import re

✅ Ambiente pronto!


#### 🔎 Como interpretar esta célula

- Carregamos `numpy`, `matplotlib` (gráficos e animações), `ipywidgets` (controles interativos) e `re` (expressões regulares).
- `mostrar_animacao(anim)` exibe as animações como um **player** (▶ ⏸ ⏭). Use o avanço quadro a quadro para estudar cada passo.

## 1. O que é Processamento de Linguagem Natural?

**Processamento de Linguagem Natural (PLN)**, ou Natural Language Processing (NLP) em inglês, é um subcampo da Inteligência Artificial que se concentra na interação entre computadores e linguagem humana. O objetivo é permitir que máquinas compreendam, interpretem e gerem linguagem natural de forma significativa e útil.

Segundo **Jurafsky e Martin** (2020), autores do livro clássico *"Speech and Language Processing"*:

> *"O Processamento de Linguagem Natural é o campo que busca construir programas computacionais que compreendam, gerem e aprendam linguagem natural."*

### 1.1 Desafios Fundamentais

A linguagem natural apresenta desafios únicos para a computação:

**Ambiguidade:** "Eu vi o homem com o telescópio" — quem tem o telescópio?

**Contexto:** "Banco" pode significar instituição financeira ou assento.

**Variabilidade:** Múltiplas formas de expressar a mesma ideia.

**Conhecimento de mundo:** "O troféu não coube na mala porque era muito grande" — o que era grande?

**Ironia e sarcasmo:** "Que dia maravilhoso!" (durante uma tempestade).

### 1.2 Breve História do PLN

- **1950 — Teste de Turing:** Alan Turing propôs avaliar inteligência de máquinas através de conversação em linguagem natural.

- **1954 — Experimento Georgetown-IBM:** Primeira demonstração de tradução automática (russo-inglês), gerando otimismo inicial excessivo.

- **1966 — ELIZA:** Joseph Weizenbaum criou chatbot que simulava psicoterapeuta usando correspondência de padrões simples, demonstrando limitações da abordagem puramente sintática.

- **1970s — Sistemas baseados em regras:** SHRDLU (Terry Winograd) entendia comandos sobre mundo de blocos. Abordagens simbólicas dominavam.

- **1980s-90s — Abordagens estatísticas:** Mudança paradigmática de regras escritas manualmente para aprendizado automático com corpus.

- **1997 — Deep Blue (IBM):** venceu o campeão mundial de xadrez Garry Kasparov. Foi um marco da IA, embora sem relação com linguagem.

- **2001 — Modelos de linguagem n-gram:** Amplamente adotados para tarefas como reconhecimento de fala e tradução.

- **2011 — IBM Watson:** venceu campeões humanos no programa de perguntas e respostas *Jeopardy!*, combinando PLN, recuperação de informação e aprendizado de máquina.

- **2013 — Word2Vec:** Mikolov et al. introduziram embeddings de palavras que capturam semântica ("rei - homem + mulher ≈ rainha").

- **2017 — Transformers:** Vaswani et al. publicaram "Attention is All You Need", revolucionando PLN com arquitetura baseada em atenção.

- **2018-2020 — Era dos modelos pré-treinados:** BERT (Google), GPT (OpenAI), que aprendem representações gerais de linguagem e são ajustados para tarefas específicas.

- **2020-presente — Modelos massivos:** GPT-3, GPT-4, LLaMA, Claude demonstram capacidades impressionantes em geração e compreensão de texto.

### 1.3 Níveis de Análise Linguística

O PLN trabalha em múltiplos níveis de abstração:

#### Fonética e Fonologia
Análise de sons da fala (mais relevante para reconhecimento de voz).

#### Morfologia
Estrutura interna das palavras: "impossibilidade" = im + possível + idade.

#### Sintaxe
Estrutura gramatical das sentenças, árvores de parsing.

#### Semântica
Significado das palavras e sentenças.

#### Pragmática
Uso da linguagem em contexto, intenções comunicativas.

#### Discurso
Relações entre sentenças, coerência e coesão textual.

### 1.4 Uma frase, seis níveis de análise

Para ver os níveis funcionando juntos, acompanhe a frase **"Os alunos não entenderam a aula de ontem."**:

| Nível | Pergunta que o nível responde | Análise da frase |
|---|---|---|
| **Morfologia** | Como cada palavra é formada? | `alunos` = alun + o (masc.) + s (plural); `entenderam` = entend + e (vogal temática) + ram (3ª pessoa do plural, pretérito perfeito) |
| **Sintaxe** | Como as palavras se agrupam? | [Os alunos]<sub>sujeito</sub> [não entenderam [a aula de ontem]<sub>objeto</sub>]<sub>predicado</sub> |
| **Semântica** | O que a frase significa? | Existe um evento *entender* que **não** aconteceu; o agente são *os alunos*, o objeto é *a aula*, o tempo é *ontem* |
| **Pragmática** | Qual a intenção de quem fala? | Dita pelo coordenador ao professor, pode ser uma **crítica** ou um **pedido** para repetir a matéria |
| **Discurso** | Como se liga às frases vizinhas? | Na frase seguinte, "**Ela** foi muito rápida", o pronome *ela* retoma *a aula* (**anáfora**) |
| **Fonologia** | Como soa? | Dita com entonação de pergunta ("Os alunos não entenderam?"), vira um **pedido de confirmação** |

> 💡 Cada técnica desta aula atua em um ou mais níveis: tokenização e *stemming* (morfologia), regex e n-gramas (forma/sintaxe superficial), TF-IDF e *embeddings* (semântica), sentimentos (semântica + pragmática). Os LLMs tentam capturar **todos ao mesmo tempo**.

### 1.5 Os tipos de ambiguidade

A **ambiguidade** é o problema central do PLN: o mesmo texto admite mais de uma interpretação, e humanos resolvem isso **sem perceber**, usando contexto e conhecimento de mundo.

| Tipo | Exemplo | As interpretações |
|---|---|---|
| **Lexical** (uma palavra, vários sentidos) | "Comi uma **manga**." / "Sujei a **manga** da camisa." | fruta × parte da roupa |
| **Categorial** (classe gramatical) | "A **casa** é amarela." / "Ele se **casa** amanhã." | substantivo × verbo |
| **Estrutural** (sintática) | "Vi o homem **com o telescópio**." | eu usei o telescópio × o homem tinha o telescópio |
| **De escopo** | "**Todos** os alunos leram **um** livro." | o mesmo livro × cada um leu um livro diferente |
| **Referencial** (anáfora) | "O João disse ao Pedro que **ele** foi aprovado." | ele = João × ele = Pedro |
| **Pragmática** | "Você sabe que horas são?" | pergunta de sim/não × pedido para dizer as horas |

**O desafio de Winograd** (Terry Winograd, 1972; Levesque, 2012) mostra que resolver referências exige **conhecimento de mundo**:

- "O troféu não coube na mala porque **ele** era muito **grande**." → *ele* = o troféu
- "O troféu não coube na mala porque **ele** era muito **pequeno**." → *ele* = a mala

Mudar **uma única palavra** muda o referente. Nenhuma regra gramatical resolve isso: é preciso saber que objetos grandes não cabem em recipientes pequenos. Por décadas esse foi considerado um teste difícil para máquinas; os LLMs atuais acertam a maioria desses casos, o que mostra quanto "conhecimento de mundo" eles absorvem do texto.

> 🗣️ **Para pensar:** quantas interpretações tem a manchete *"Polícia prende homem com faca na rua"*? (quem está com a faca? a faca estava na rua?)

### 1.6 🧪 ELIZA: o primeiro *chatbot* (1966) em poucas linhas

A **ELIZA**, de Joseph Weizenbaum (MIT), simulava um psicoterapeuta usando apenas **regras de casamento de padrões**: procura um padrão na frase do usuário, **reflete os pronomes** (eu → você, meu → seu) e devolve uma pergunta pronta. Não há nenhum "entendimento". Mesmo assim, muitos usuários atribuíram emoções ao programa, o que ficou conhecido como **efeito ELIZA**.

É um **sistema de produção** (Aula 07): uma lista ordenada de regras **SE padrão ENTÃO resposta**, testadas de cima para baixo, com uma regra "pega-tudo" no final.

In [3]:
# 🧪 Mini-ELIZA: regras SE (padrão) ENTÃO (resposta) + reflexão de pronomes
import random

REGRAS_ELIZA = [   # (padrão regex, respostas possíveis). A ORDEM importa: a primeira que casar vence.
    (r'\bpreciso de (.+)',        ['Por que você precisa de {0}?', 'Ter {0} resolveria mesmo o seu problema?']),
    (r'\beu sou (.+)',            ['Há quanto tempo você é {0}?', 'Como você se sente sendo {0}?']),
    (r'\beu estou (.+)',          ['Por que você está {0}?', 'Há quanto tempo você está {0}?']),
    (r'\b(?:minha|meu) (\w+)',    ['Fale mais sobre isso: seu/sua {0}.', 'Por que você mencionou seu/sua {0}?']),
    (r'\b(sempre|nunca)\b',       ['"{0}" é uma palavra forte. Pode dar um exemplo específico?']),
    (r'\bporque (.+)',            ['Essa é a verdadeira razão?', 'Que outras razões poderiam existir?']),
    (r'\?$',                      ['Por que você pergunta isso?', 'O que você acha?']),
    (r'.*',                       ['Entendo. Continue.', 'Como isso faz você se sentir?']),   # regra pega-tudo
]
REFLEXOES = {'eu': 'você', 'meu': 'seu', 'minha': 'sua', 'meus': 'seus', 'minhas': 'suas',
             'me': 'te', 'mim': 'você', 'estou': 'está', 'sou': 'é', 'comigo': 'com você'}

def refletir(trecho):
    # Troca a pessoa do discurso: "minha prova" -> "sua prova"
    return ' '.join(REFLEXOES.get(p, p) for p in trecho.split())

def eliza(frase, rng=random.Random(1966)):
    frase = frase.lower().strip(' .!')
    for i, (padrao, respostas) in enumerate(REGRAS_ELIZA):
        m = re.search(padrao, frase)
        if m:
            grupos = [refletir(g) for g in m.groups()]
            return rng.choice(respostas).format(*grupos), i

conversa = ['Eu estou cansado com as provas.',
            'Preciso de ajuda com minha monografia.',
            'Meu orientador nunca responde os e-mails.',
            'Porque ele tem muitos alunos.',
            'Você acha que vou passar?',
            'O céu é azul.']
for frase in conversa:
    resposta, regra = eliza(frase)
    print(f'👤 {frase}\n🤖 {resposta}   [regra {regra}: {REGRAS_ELIZA[regra][0]}]\n')

# ✍️ Experimente: print(eliza('Eu sou muito ansioso')[0])

👤 Eu estou cansado com as provas.
🤖 Por que você está cansado com as provas?   [regra 2: \beu estou (.+)]

👤 Preciso de ajuda com minha monografia.
🤖 Ter ajuda com sua monografia resolveria mesmo o seu problema?   [regra 0: \bpreciso de (.+)]

👤 Meu orientador nunca responde os e-mails.
🤖 Fale mais sobre isso: seu/sua orientador.   [regra 3: \b(?:minha|meu) (\w+)]

👤 Porque ele tem muitos alunos.
🤖 Essa é a verdadeira razão?   [regra 5: \bporque (.+)]

👤 Você acha que vou passar?
🤖 Por que você pergunta isso?   [regra 6: \?$]

👤 O céu é azul.
🤖 Como isso faz você se sentir?   [regra 7: .*]



#### 🔎 Como interpretar o resultado

- **Cada resposta veio de uma regra** (o número entre colchetes). Não existe modelo de linguagem, nem dicionário, nem significado: só **regex + troca de pronomes + respostas prontas**.
- **Reflexão de pronomes:** "Preciso de ajuda com **minha** monografia" → "Ter ajuda com **sua** monografia...". O trecho capturado pelo grupo `(.+)` é devolvido com a pessoa trocada. É isso que dá a **ilusão** de que o programa "ouviu" o usuário.
- **A ordem das regras importa:** em "Meu orientador **nunca** responde", a regra do `meu` (3) vem antes da regra do `nunca` (4), então vence. Reordenar as regras muda a "personalidade" do *chatbot*. É a **resolução de conflitos** dos sistemas de produção (Aula 07).
- **As falhas mostram a falta de compreensão:**
  - "Fale mais sobre isso: seu/sua orientador." O programa não sabe que *orientador* é masculino: não há **morfologia**;
  - "Você acha que vou passar?" recebe uma resposta genérica de pergunta; o programa não tem a menor ideia do que é "passar";
  - "O céu é azul." cai na regra **pega-tudo**: sem padrão reconhecido, a ELIZA "enrola".
- **💡 Conceito-chave (efeito ELIZA):** humanos **atribuem compreensão** a qualquer sistema que responda de forma plausível. Weizenbaum ficou assustado quando sua secretária pediu para ficar a sós com o programa. A pergunta *"responder bem é o mesmo que entender?"* continua atual com os LLMs.

> ✍️ **Experimente:** adicione uma regra para `r'\b(?:estou|me sinto) (triste|feliz|ansioso)'` e outra para cumprimentos (`oi`, `olá`). Onde você precisa colocá-las na lista para funcionarem?

### 1.7 💬 Converse com a ELIZA

A célula anterior **simulou** uma conversa com frases fixas. Agora a conversa é **de verdade**: você escreve o que quiser e a ELIZA responde na hora. Esta versão é um pouco mais completa e se aproxima do *script* **DOCTOR** original de Weizenbaum:

| Recurso | Como funciona | No original de 1966? |
|---|---|:-:|
| **Palavras-chave em ordem de prioridade** | ~20 regras testadas de cima para baixo; a primeira que casar responde | ✅ (lá, cada palavra-chave tinha um *rank*) |
| **Reflexão de pronomes** | "minha prova" → "sua prova", "estou" → "está" | ✅ |
| **Respostas em rodízio** | cada regra tem várias respostas usadas em sequência, para não repetir sempre a mesma | ✅ |
| **Memória** | frases com "meu/minha" são guardadas e retomadas mais tarde, quando nenhuma regra casar | ✅ |
| **Regra pega-tudo** | sem padrão reconhecido, devolve uma frase genérica ("Entendo. Continue.") | ✅ |

**Como usar:** execute a célula, escreva na caixa e tecle **Enter** (ou clique em **Enviar**). Com **"mostrar a regra usada"** marcado, aparece embaixo de cada resposta qual regra disparou e o que foi capturado: é o "modo professor", para mostrar à turma que não há nada além de regras. Diga **tchau** para encerrar.

> 💡 **Sugestão para a sala:** peça para um aluno conversar por 2 minutos **sem** o modo professor e depois pergunte se a ELIZA "entendeu". Em seguida, ligue o modo professor e repita as mesmas frases.

In [ ]:
# 💬 ELIZA interativa: um chatbot de 1966 feito só de regras (nenhum aprendizado, nenhum dado de treino)
import html

class ElizaPT:
    # Reflexão: troca a pessoa do discurso no trecho capturado ("minha prova" -> "sua prova")
    REFLEXOES = {'eu': 'você', 'me': 'te', 'mim': 'você', 'comigo': 'com você', 'meu': 'seu', 'minha': 'sua',
                 'meus': 'seus', 'minhas': 'suas', 'sou': 'é', 'estou': 'está', 'tenho': 'tem', 'fui': 'foi',
                 'quero': 'quer', 'preciso': 'precisa', 'sinto': 'sente', 'acho': 'acha', 'gosto': 'gosta',
                 'odeio': 'odeia', 'consigo': 'consegue', 'posso': 'pode', 'você': 'eu', 'te': 'me'}
    # Quem digita sem acento também é entendido
    SEM_ACENTO = {'voce': 'você', 'nao': 'não', 'ola': 'olá', 'mae': 'mãe', 'irma': 'irmã', 'irmao': 'irmão',
                  'avo': 'avó', 'maquina': 'máquina', 'robo': 'robô', 'tambem': 'também', 'ninguem': 'ninguém'}
    SENT_NEG = r'(triste|deprimid[oa]s?|cansad[oa]s?|ansios[oa]s?|nervos[oa]s?|preocupad[oa]s?|sozinh[oa]s?|estressad[oa]s?|chatead[oa]s?|com medo|mal)'
    SENT_POS = r'(feliz|alegre|animad[oa]s?|content[ea]s?|bem|tranquil[oa]s?|orgulhos[oa]s?)'

    # (nome, padrão, respostas). A ORDEM é a prioridade: a primeira regra que casar responde.
    REGRAS = [
        ('despedida',    r'\b(tchau|adeus|até logo|até mais|sair)\b',
                         ['Até logo. Foi bom conversar com você.']),
        ('cumprimento',  r'^(oi|olá|bom dia|boa tarde|boa noite|e aí)\b',
                         ['Olá. Como você está se sentindo hoje?', 'Oi! Sobre o que você gostaria de conversar?']),
        ('desculpa',     r'\b(desculp\w*|perdão)\b',
                         ['Não precisa pedir desculpas.', 'Como você se sente quando pede desculpas?']),
        ('sentimento -', r'\b(?:estou|me sinto|ando|fico|tô) (?:muito |meio |tão |bem |um pouco )?' + SENT_NEG + r'\b',
                         ['Sinto muito que você esteja {0}. O que faz você se sentir assim?',
                          'Há quanto tempo você está {0}?', 'Você acha que conversar ajuda quando você está {0}?']),
        ('sentimento +', r'\b(?:estou|me sinto|ando|fico|tô) (?:muito |meio |tão |bem )?' + SENT_POS + r'\b',
                         ['Que bom! O que deixou você {0}?', 'Como é para você se sentir {0}?']),
        ('preciso',      r'\bpreciso (?:de )?(.+)',
                         ['Por que você precisa de {0}?', 'Ter {0} resolveria mesmo o seu problema?',
                          'Você tem certeza de que precisa de {0}?']),
        ('quero',        r'\bquero (.+)',
                         ['O que significaria para você conseguir {0}?', 'Por que você quer {0}?']),
        ('não consigo',  r'\bnão consigo (.+)',
                         ['Como você sabe que não consegue {0}?', 'O que aconteceria se você conseguisse {0}?']),
        ('eu acho',      r'\b(?:eu )?acho que (.+)',
                         ['Você tem certeza de que {0}?', 'Por que você acha que {0}?']),
        ('você é',       r'\bvocê é (.+)',
                         ['Por que interessa a você se eu sou {0}?', 'Você gostaria que eu fosse {0}?']),
        ('eu sou',       r'\beu sou (.+)',
                         ['Há quanto tempo você é {0}?', 'Como você se sente sendo {0}?', 'Você gosta de ser {0}?']),
        ('eu estou',     r'\b(?:eu )?estou (.+)',
                         ['Por que você está {0}?', 'Você acredita que é normal estar {0}?']),
        ('máquinas',     r'\b(computador\w*|máquina\w*|robô\w*|programa|ia|chatbot|inteligência artificial)\b',
                         ['Computadores preocupam você?', 'Por que você mencionou máquinas?',
                          'Você acha que uma máquina pode entender você?']),
        ('família',      r'\b(mãe|pai|irmã|irmão|família|filho|filha|esposa|marido|namorad[oa]|avó|avô)\b',
                         ['Fale mais sobre sua família.', 'Como é o seu relacionamento com essa pessoa ({0})?',
                          'Quem mais da sua família vem à sua mente?']),
        ('meu/minha',    r'\b(meu|minha|meus|minhas) (.+)',
                         ['Por que você diz que {0} {1}?', 'Fale mais sobre isso: {0} {1}.']),
        ('generalização', r'\b(sempre|nunca|ninguém|todo mundo|todos)\b',
                         ['Você consegue pensar em um exemplo específico?', '"{0}" é uma palavra forte. É assim mesmo?']),
        ('porque',       r'\bporque (.+)',
                         ['Essa é a verdadeira razão?', 'Que outras razões poderiam existir?']),
        ('sonho',        r'\bsonh\w+',
                         ['O que esse sonho sugere para você?', 'Você sonha com isso com frequência?']),
        ('sim',          r'^(sim|claro|com certeza)\b',
                         ['Você parece bem seguro.', 'Entendo. Pode explicar melhor?']),
        ('não',          r'^não\b',
                         ['Por que não?', 'Você está dizendo não só para ser do contra?']),
        ('pergunta',     r'\?$',
                         ['Por que você pergunta isso?', 'O que você acha?', 'Que resposta agradaria mais a você?']),
    ]
    GENERICAS = ['Entendo. Continue.', 'Fale mais sobre isso.', 'Como isso faz você se sentir?',
                 'Interessante. Por que você diz isso?', 'Hum... e o que mais?']

    def __init__(self):
        self.uso = {}          # rodízio: quantas vezes cada regra já respondeu
        self.memoria = []      # frases guardadas para retomar depois
        self.encerrou = False

    def _refletir(self, trecho):
        return ' '.join(self.REFLEXOES.get(p, p) for p in trecho.strip(' .,!?;').split())

    def _proxima(self, nome, respostas):
        i = self.uso.get(nome, 0)
        self.uso[nome] = i + 1
        return respostas[i % len(respostas)]

    def responder(self, frase):
        texto = ' '.join(self.SEM_ACENTO.get(p, p) for p in frase.lower().strip().split())
        for nome, padrao, respostas in self.REGRAS:
            m = re.search(padrao, texto)
            if m:
                grupos = [self._refletir(g) for g in m.groups()]
                if nome == 'meu/minha':
                    self.memoria.append(' '.join(grupos))
                if nome == 'despedida':
                    self.encerrou = True
                return self._proxima(nome, respostas).format(*grupos), {'regra': nome, 'padrão': padrao, 'capturado': grupos}
        if self.memoria and self.uso.get('genérica', 0) % 2 == 1:          # de vez em quando, retoma a memória
            lembranca = self.memoria.pop(0)
            self.uso['genérica'] = self.uso.get('genérica', 0) + 1
            return f'Antes você disse que {lembranca}. Vamos voltar a isso?', {'regra': 'memória', 'padrão': '—', 'capturado': [lembranca]}
        return self._proxima('genérica', self.GENERICAS), {'regra': 'pega-tudo', 'padrão': '—', 'capturado': []}


def chat_eliza():
    bot = ElizaPT()
    tela = widgets.Output(layout=widgets.Layout(border='1px solid #bbb', height='340px', overflow_y='auto', padding='6px 10px'))
    caixa = widgets.Text(placeholder='Escreva aqui e tecle Enter (ex.: Oi, estou nervoso com a prova)',
                         continuous_update=False, layout=widgets.Layout(width='68%'))
    enviar = widgets.Button(description='Enviar', button_style='primary')
    recomecar = widgets.Button(description='Recomeçar')
    modo_prof = widgets.Checkbox(value=True, description='mostrar a regra usada (modo professor)', indent=False)

    def bolha(quem, texto, cor, alinhamento):
        return (f'<div style="text-align:{alinhamento};margin:4px 0"><span style="display:inline-block;max-width:80%;'
                f'background:{cor};color:#111;padding:6px 10px;border-radius:10px"><b>{quem}:</b> {html.escape(texto)}</span></div>')

    def falar(_=None):
        frase = caixa.value.strip()
        if not frase:
            return
        caixa.value = ''
        resposta, info = bot.responder(frase)
        with tela:
            display(HTML(bolha('Você', frase, '#dbe8fb', 'right')))
            display(HTML(bolha('ELIZA', resposta, '#eeeeee', 'left')))
            if modo_prof.value:
                capt = ', '.join(f'"{g}"' for g in info['capturado']) or '—'
                display(HTML(f'<div style="font:12px monospace;color:#8a5a00;margin:0 0 8px 6px">🔍 regra: <b>{info["regra"]}</b>'
                             f' · padrão: {html.escape(info["padrão"][:70])} · capturado (refletido): {html.escape(capt)}</div>'))
            if bot.encerrou:
                display(HTML('<div style="color:#888;margin:6px">— conversa encerrada. Clique em Recomeçar. —</div>'))

    def reiniciar(_=None):
        nonlocal bot
        bot = ElizaPT()
        tela.clear_output()
        with tela:
            display(HTML(bolha('ELIZA', 'Olá, eu sou a ELIZA. Como você está se sentindo hoje?', '#eeeeee', 'left')))

    caixa.observe(falar, 'value')        # Enter (ou sair da caixa) envia a frase
    enviar.on_click(falar)
    recomecar.on_click(reiniciar)
    reiniciar()
    display(widgets.VBox([tela, widgets.HBox([caixa, enviar, recomecar]), modo_prof]))


def conversar_eliza():
    # Versão em texto puro (use se os controles interativos não aparecerem no seu ambiente)
    bot = ElizaPT()
    print('ELIZA: Olá, eu sou a ELIZA. Como você está se sentindo hoje?  (digite "tchau" para sair)')
    while not bot.encerrou:
        resposta, info = bot.responder(input('Você: '))
        print(f'ELIZA: {resposta}   [regra: {info["regra"]}]')


chat_eliza()
# conversar_eliza()   # ← alternativa sem widgets: descomente esta linha e comente a de cima

#### 🔎 Como conduzir e interpretar a conversa

Um roteiro que funciona bem em sala (as respostas abaixo são as que esta ELIZA dá, nesta ordem):

| Você escreve | A ELIZA responde | Regra | O que mostrar à turma |
|---|---|---|---|
| Oi | Olá. Como você está se sentindo hoje? | cumprimento | Resposta pronta, sem nada capturado |
| Estou nervoso com a prova de cálculo | Sinto muito que você esteja nervoso. O que faz você se sentir assim? | sentimento − | Capturou só `nervoso`; "prova de cálculo" foi **ignorada** |
| Minha professora passa muita matéria | Por que você diz que sua professora passa muita matéria? | meu/minha | Reflexão `minha → sua` e a frase foi para a **memória** |
| Eu preciso de mais tempo para estudar | Por que você precisa de mais tempo para estudar? | preciso | Devolve o próprio texto do usuário como pergunta |
| Você é um robô? | Por que interessa a você se eu sou um robô? | você é | Desvia a pergunta: nunca responde sobre si mesma |
| Quanto é 2 + 2? | Por que você pergunta isso? | pergunta | **Não sabe fazer conta**: só viu o `?` |
| Ok | Entendo. Continue. | pega-tudo | Nenhuma regra casou |
| Hmm | Antes você disse que sua professora passa muita matéria. Vamos voltar a isso? | memória | Parece atenção ao que foi dito, mas é uma fila de frases |

**Pontos para discutir:**

- **Nada é entendido.** A ELIZA não sabe o que é prova, cálculo, professora ou robô. Ela casa **padrões de superfície** e **devolve as palavras do próprio usuário**. Quem dá sentido à conversa é a pessoa.
- **Por que funciona tão bem?** Weizenbaum escolheu o papel de psicoterapeuta rogeriano de propósito: nesse estilo de terapia, devolver perguntas é **esperado**. O contexto esconde a falta de conhecimento.
- **Onde quebra:**
  - perguntas factuais ("quanto é 2 + 2?") e frases sem palavra-chave;
  - mais de uma ideia na mesma frase: em "Meu pai nunca me escuta", só a regra de maior prioridade (família) responde, e o "nunca me escuta" se perde;
  - concordância verbal: "Acho que vou reprovar" vira "Você tem certeza de que **vou** reprovar?". A reflexão troca pronomes de uma lista, mas não sabe conjugar verbos;
  - ironia e sarcasmo: "Adorei tirar zero" não tem nenhuma palavra-chave negativa.
- **Peça para a turma "quebrar" a ELIZA:** em poucos minutos alguém descobre que ela nunca responde nada de concreto. Isso leva direto à próxima seção.

### 1.8 ELIZA × IAs generativas de hoje (ChatGPT, Claude, Gemini)

Por fora, as duas coisas parecem iguais: você escreve, o programa responde. Por dentro, são **opostos**:

| | **ELIZA (1966)** | **IA generativa / LLM (2022–hoje)** |
|---|---|---|
| **Como produz a resposta** | Procura uma **regra** que case com a frase e preenche uma **resposta pronta** | **Gera** a resposta token por token, escolhendo a cada passo o próximo token mais provável (Seção 4.5) |
| **De onde vem o comportamento** | Escrito **à mão** pelo programador (~20 regras aqui; o DOCTOR original tinha algumas centenas de linhas) | **Aprendido** de trilhões de tokens de texto, depois ajustado com instruções e feedback humano (Seção 4.8) |
| **Tamanho** | Algumas dezenas de regras | Bilhões de **parâmetros** (números ajustados no treino) |
| **Conhecimento de mundo** | Nenhum. Não sabe o que é uma prova nem quanto é 2 + 2 | Muito, absorvido dos textos: fatos, gramática, código, estilos. Mas pode estar **desatualizado** ou **errado** |
| **Uso do contexto** | Só a frase atual (mais uma pequena fila de memória) | Toda a conversa, até centenas de milhares de tokens, via **atenção** (Seção 4.7) |
| **Frases novas** | Nunca: só monta frases a partir de modelos fixos | Sempre: produz textos que nunca existiram |
| **Língua e tarefas** | Uma língua, uma tarefa (o *script* DOCTOR) | Muitas línguas e tarefas (resumir, traduzir, programar, explicar) sem trocar de programa |
| **Quando não sabe** | "Entendo. Continue." (**enrola** de forma visível) | Pode **alucinar**: inventar uma resposta fluente e convincente, porém falsa |
| **Explicabilidade** | Total: dá para apontar a regra exata (modo professor) | Baixa: a resposta sai de bilhões de multiplicações; não há "a regra" a apontar |
| **Custo** | Roda em qualquer computador desde 1966 | Treino em milhares de GPUs; cada resposta também exige hardware considerável |
| **Paradigma de IA** | **Simbólico** (regras, Aulas 05 e 07) | **Conexionista / estatístico** (redes neurais, Aulas 06 e 11) |

**O que eles têm em comum (e por que a ELIZA ainda importa):**

1. **O efeito ELIZA continua valendo.** Atribuímos compreensão, intenção e até sentimentos a qualquer sistema que responda de forma fluente. Com os LLMs o efeito é muito mais forte, porque as respostas são muito melhores.
2. **Nenhum dos dois "entende" do jeito que uma pessoa entende**, pelo menos não de forma garantida. A ELIZA claramente não entende; os LLMs **modelam** a linguagem tão bem que a pergunta "isso é compreensão?" virou um debate sério, sem resposta consensual.
3. **Ambos podem ser usados como "terapeutas"**, e o alerta de Weizenbaum continua atual. Ele passou o resto da carreira (*Computer Power and Human Reason*, 1976) argumentando que certas decisões e relações humanas não deveriam ser delegadas a máquinas.

> 🧪 **Experimento para a sala:** escreva a mesma frase para a ELIZA (acima) e para um LLM, por exemplo *"Estou nervoso com a prova de cálculo amanhã. O que eu faço?"*.
> - A ELIZA **devolve** a frase como pergunta ("Sinto muito que você esteja nervoso...").
> - O LLM **responde de verdade**: sugere revisar exercícios, dormir bem, fazer simulados... conteúdo que **não estava** na sua frase e que veio do que ele aprendeu.
>
> Depois pergunte: *"quanto é 17 × 23?"* e *"quem ganhou a Copa de 1970?"*. A ELIZA desvia; o LLM responde, e vale conferir se está certo. Essa diferença é o caminho percorrido nesta aula: de **regras** para **contagens**, para **aprendizado**, para **atenção**.

## 2. Pré-processamento de Texto

Antes de aplicar algoritmos de PLN, o texto precisa ser preparado. Etapas comuns:

### 2.1 Tokenização

**Tokenização** divide texto em unidades menores (tokens): palavras, subpalavras ou caracteres.

**Desafios:**
- Pontuação: "Dr. Silva" deve ser 2 ou 3 tokens?
- Contrações: "não é" → "não" + "é" ou manter junto?
- Compostos: "guarda-chuva" é 1 ou 3 tokens?

### 2.2 Normalização

- **Lowercasing:** "Casa" → "casa"
- **Remoção de acentos:** "São Paulo" → "sao paulo" (depende da aplicação)
- **Stemming:** Reduz palavras à raiz: "correndo", "corrida", "correr" → "corr"
- **Lemmatização:** Reduz à forma canônica: "correndo" → "correr"

### 2.3 Remoção de Stopwords

**Stopwords** são palavras muito frequentes com pouco valor semântico: "o", "a", "de", "para", etc.

### 2.4 Implementação de Tokenizador Simples

In [ ]:
import re
from collections import Counter
from typing import List, Set

class TokenizadorSimples:
    """
    Tokenizador básico para texto em português.
    """
    
    def __init__(self, lowercase=True, remover_pontuacao=True):
        self.lowercase = lowercase
        self.remover_pontuacao = remover_pontuacao
        
        # Stopwords comuns em português
        self.stopwords = {
            'o', 'a', 'os', 'as', 'um', 'uma', 'de', 'do', 'da', 'dos', 'das',
            'em', 'no', 'na', 'nos', 'nas', 'por', 'para', 'com', 'sem', 'sob',
            'e', 'ou', 'mas', 'porque', 'que', 'qual', 'quando', 'onde', 'como',
            'é', 'são', 'foi', 'ser', 'estar', 'ter', 'fazer', 'muito', 'mais'
        }
    
    def tokenizar(self, texto: str) -> List[str]:
        """
        Tokeniza texto em palavras.
        """
        if self.lowercase:
            texto = texto.lower()
        
        if self.remover_pontuacao:
            # Remove pontuação mas mantém letras, números e espaços
            texto = re.sub(r'[^\w\s]', '', texto)
        
        # Divide por espaços em branco
        tokens = texto.split()
        
        return tokens
    
    def remover_stopwords(self, tokens: List[str]) -> List[str]:
        """
        Remove stopwords da lista de tokens.
        """
        return [token for token in tokens if token not in self.stopwords]
    
    def stemming_simples(self, palavra: str) -> str:
        """
        Stemming muito simplificado (remove sufixos comuns).
        Nota: Para uso real, use biblioteca como PyStemmer ou NLTK.
        """
        sufixos = ['mente', 'ação', 'ador', 'ante', 'ância', 'ência', 
                   'ismo', 'ista', 'oso', 'osa', 'ável', 'ível',
                   'ando', 'endo', 'indo', 'ada', 'ida', 'ado', 'ido']
        
        for sufixo in sorted(sufixos, key=len, reverse=True):
            if palavra.endswith(sufixo) and len(palavra) > len(sufixo) + 2:
                return palavra[:-len(sufixo)]
        
        return palavra

# Exemplo de uso
print("="*60)
print("TOKENIZAÇÃO E PRÉ-PROCESSAMENTO")
print("="*60)

texto = """O Processamento de Linguagem Natural é uma área fascinante da 
Inteligência Artificial. Ela permite que computadores compreendam e gerem 
texto de forma automática, possibilitando aplicações como chatbots, 
tradução automática e análise de sentimentos."""

tokenizador = TokenizadorSimples()

print("\nTexto original:")
print(texto)

# Tokenização
tokens = tokenizador.tokenizar(texto)
print(f"\n\nTokens ({len(tokens)}):")
print(tokens[:15], "...")

# Remoção de stopwords
tokens_sem_stop = tokenizador.remover_stopwords(tokens)
print(f"\n\nTokens sem stopwords ({len(tokens_sem_stop)}):")
print(tokens_sem_stop)

# Stemming
tokens_stem = [tokenizador.stemming_simples(t) for t in tokens_sem_stop]
print(f"\n\nTokens com stemming:")
print(tokens_stem)

# Frequência de palavras
freq = Counter(tokens_sem_stop)
print(f"\n\nPalavras mais frequentes:")
for palavra, count in freq.most_common(5):
    print(f"  {palavra}: {count}")

print("\n" + "="*60)

#### 🔎 Como interpretar o resultado

| Etapa | Resultado | O que observar |
|---|---|---|
| **Tokenização** | 33 tokens | Tudo em minúsculas e sem pontuação: "Artificial." virou `artificial` |
| **Sem stopwords** | 22 tokens | Saíram `o`, `de`, `é`, `uma`, `da`, `que`, `e`, `como`... (**1/3 do texto!**) |
| **Stemming** | — | `fascinante → fascin`, `inteligência → intelig`, `possibilitando → possibilit` |
| **Frequência** | `automática: 2` | É a única palavra repetida; o resto aparece uma vez cada |

- **Por que as stopwords são 1/3 do texto?** Palavras funcionais (artigos, preposições, conjunções) são as **mais frequentes** de qualquer língua (veja a Lei de Zipf na Seção 2.7). Elas carregam estrutura gramatical, mas pouco **conteúdo**.
- **Limitações visíveis na saída:**
  - `ela` **não** foi removida: não está na nossa lista de stopwords. Toda lista de stopwords é uma **escolha** e depende da aplicação;
  - O *stemmer* não mexeu em `compreendam`, `gerem`, `aplicações`, `automática`. Ele só conhece alguns sufixos, e `aplicações` (plural de `-ação`) escapou. Um *stemmer* real para o português, como o **RSLP** (Orengo e Huyck, 2001), tem cerca de 200 regras organizadas em etapas (plural, feminino, aumentativo, verbo...).
- **💡 Conceito-chave:** o pré-processamento **reduz o vocabulário**, juntando variantes da mesma ideia, mas sempre **perde informação**. Por isso deve ser ajustado à tarefa. Para análise de sentimentos, por exemplo, remover "não" seria um desastre.

**Observações:**

1. **Tokenização simples:** Dividir por espaços funciona razoavelmente para português, mas não lida com casos complexos como URLs, emails ou números com separadores.

2. **Stemming vs. Lemmatização:** Stemming é mais rápido mas menos preciso (pode gerar raízes não-palavras). Lemmatização preserva palavras válidas mas requer conhecimento morfológico.

3. **Stopwords:** A lista deve ser ajustada para o domínio. Em análise de sentimentos, "não" é crucial e não deve ser removida.

4. **Normalização:** Lowercase é útil para reduzir vocabulário, mas perde informação ("Apple" empresa vs "apple" fruta).

### 2.4.1 *Stemming* × lematização na prática

O nosso `stemming_simples` só conhece alguns sufixos. Vamos compará-lo com o **RSLP** (*Removedor de Sufixos da Língua Portuguesa*, Orengo e Huyck, 2001), que aplica cerca de 200 regras em 8 etapas (plural → feminino → advérbio → aumentativo/diminutivo → nome → verbo → vogal temática → acentos), e com a **lematização**, que devolve a **forma de dicionário** (o *lema*).

| | *Stemming* | Lematização |
|---|---|---|
| **Como funciona** | Corta sufixos com regras | Consulta um dicionário morfológico + classe gramatical |
| **Saída** | Um **radical**, que pode não ser palavra (`estud`) | Uma **palavra válida** (`estudar`) |
| **Velocidade** | Muito rápida | Mais lenta |
| **Precisa de contexto?** | Não | Às vezes (`fui` → *ser* ou *ir*?) |
| **Ferramentas** | NLTK (`RSLPStemmer`, `SnowballStemmer('portuguese')`) | spaCy (`token.lemma_`), Stanza |

In [ ]:
# Stemming × lematização: nosso stemmer, o RSLP (NLTK) e um mini-dicionário de lemas
import nltk
try:
    nltk.download('rslp', quiet=True)
    from nltk.stem import RSLPStemmer
    rslp = RSLPStemmer().stem
except Exception as erro:
    print(f'⚠️ RSLP indisponível ({erro.__class__.__name__}); a coluna RSLP ficará vazia.')
    rslp = lambda p: '—'

# Lematizador "de brinquedo": um dicionário forma → lema (spaCy e Stanza usam dicionários enormes + contexto)
LEMAS = {'alunos': 'aluno', 'alunas': 'aluno', 'estudando': 'estudar', 'estudou': 'estudar', 'estudam': 'estudar',
         'fui': 'ser / ir ❓', 'era': 'ser', 'sou': 'ser', 'melhores': 'melhor', 'pedreiro': 'pedreiro',
         'pedras': 'pedra', 'felicidade': 'felicidade', 'felizmente': 'felizmente', 'casinha': 'casa',
         'cantaríamos': 'cantar', 'organização': 'organização', 'organizado': 'organizar'}

print(f'{"palavra":<14}{"nosso stemmer":<16}{"RSLP":<12}{"lema"}')
print('-' * 56)
for p, lema in LEMAS.items():
    print(f'{p:<14}{tokenizador.stemming_simples(p):<16}{rslp(p):<12}{lema}')

#### 🔎 Como interpretar o resultado

- **O RSLP agrupa muito mais variantes:** `alunos/alunas → alun`, `estudando/estudou/estudam → estud`. O nosso *stemmer* não toca em `alunos`, `estudou` ou `estudam`, porque não conhece os sufixos de plural e de flexão verbal.
- **Radicais não são palavras:** `alun`, `estud`, `cant` não existem no dicionário. Para um **buscador** isso não importa (só precisamos que variantes caiam no mesmo "balde"); para mostrar o resultado a um usuário, sim.
- **Erros de *stemming*:**
  - ***Overstemming*** (junta o que não devia): `pedras` e `pedreiro` viram `pedr`, mas *pedreiro* é uma profissão, não uma pedra. `casinha` vira `cas`, o mesmo radical de *casa*, mas também de *casar* e *casamento*;
  - ***Understemming*** (separa o que devia juntar): `felicidade → felic` e `felizmente → feliz` ficam diferentes, embora venham da mesma ideia.
- **Formas irregulares só a lematização resolve:** `fui`, `era`, `sou` são formas de *ser* (e `fui`, também de *ir*!). Nenhuma regra de corte de sufixo chega a `ser` a partir de `fui`. E para decidir entre *ser* e *ir* é preciso **contexto**: "fui ao mercado" × "fui aluno dele". É por isso que lematizadores reais (spaCy, Stanza) primeiro fazem **etiquetagem morfossintática** (*POS tagging*).
- **💡 Conceito-chave:** *stemming* é uma **heurística barata** (bom para busca e classificação); lematização é uma **análise linguística** (melhor para tarefas que mostram ou interpretam as palavras). Os modelos modernos com subpalavras (BPE, Seção 2.8) dispensam as duas: aprendem sozinhos que `estud-` é um pedaço recorrente.

### 2.5 🎛️ Pipeline de pré-processamento interativo

Edite o texto e ligue/desligue cada etapa. Observe duas medidas: o número de **tokens** (ocorrências) e o de **tipos** (palavras distintas, o tamanho do vocabulário).

In [ ]:
# 🎛️ Explore cada etapa do pré-processamento (edite o texto e marque/desmarque as opções)
def explorar_preprocessamento(texto, minusculas=True, sem_pontuacao=True, sem_stopwords=True, stemming=False):
    tk = TokenizadorSimples(lowercase=minusculas, remover_pontuacao=sem_pontuacao)
    tokens = tk.tokenizar(texto)
    etapas = [('1. tokenização', tokens)]
    if sem_stopwords:
        tokens = tk.remover_stopwords(tokens); etapas.append(('2. sem stopwords', tokens))
    if stemming:
        tokens = [tk.stemming_simples(t) for t in tokens]; etapas.append(('3. stemming', tokens))
    for nome, toks in etapas:
        print(f'{nome:<18} | {len(toks):3d} tokens | {len(set(toks)):3d} tipos (palavras distintas)')
        print(f'{"":<18} | {toks}\n')

interact(explorar_preprocessamento,
         texto=widgets.Textarea(value='Os alunos estavam estudando. O aluno estudou e as alunas estudam muito!',
                                layout=widgets.Layout(width='90%', height='60px')));

#### 🔎 Como usar e interpretar

Com o texto padrão (*"Os alunos estavam estudando. O aluno estudou e as alunas estudam muito!"*):

- **Tokenização:** 12 tokens e 12 tipos. Todas as palavras são diferentes **como cadeias**, embora `aluno/alunos/alunas` e `estudando/estudou/estudam` representem só **duas ideias**.
- **Sem stopwords:** sobram 7 tokens, todos de conteúdo.
- **Marque `stemming`:** apenas `estudando → estud` muda. `alunos`, `alunas`, `estudou` e `estudam` **continuam diferentes**, porque nosso *stemmer* não conhece os sufixos `-os`, `-as`, `-ou`, `-am`. Com um *stemmer* completo, todos virariam `alun` e `estud`, e o vocabulário cairia de 7 para **2 tipos**.

**Experimente:**
1. Desmarque `minusculas` e escreva "Casa casa CASA": são 3 tipos diferentes para o computador.
2. Desmarque `sem_pontuacao`: `estudando.` e `estudando` viram tokens diferentes.
3. Cole uma frase com "não" e veja que ele **sobrevive** às stopwords (não está na lista) — ainda bem!

- **💡 Conceito-chave, tokens × tipos:** *tokens* são as ocorrências; *tipos* são as palavras distintas (o vocabulário). O objetivo da normalização é **reduzir os tipos** sem perder o que importa para a tarefa.

### 2.6 Expressões regulares: extraindo informação estruturada

Antes de qualquer modelo estatístico, muitas tarefas de PLN se resolvem com **expressões regulares** (*regex*): padrões que descrevem conjuntos de cadeias. Elas são, na prática, **sistemas de produção sobre texto** (Aula 07) e descendem diretamente dos trabalhos de Kleene e Chomsky sobre linguagens formais.

In [ ]:
import re

texto_rede_social = """Oi @maria_silva! O evento #IA2026 será em 15/03/2026 às 19h30.
Inscrições em https://exemplo.com.br/evento ou pelo e-mail contato@exemplo.com.br.
Ingresso: R$ 49,90 (meia: R$ 24,95). Dúvidas? Ligue (11) 98765-4321. #PLN #Python"""

padroes = {
    'menções':      r'(?<!\w)@\w+',
    'hashtags':     r'#\w+',
    'URLs':         r'https?://[^\s]+[^\s.,!?)]',
    'e-mails':      r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+',
    'datas':        r'\b\d{2}/\d{2}/\d{4}\b',
    'horários':     r'\b\d{1,2}h\d{2}\b',
    'valores (R$)': r'R\$\s?\d+(?:\.\d{3})*,\d{2}',
    'telefones':    r'\(\d{2}\)\s?\d{4,5}-\d{4}',
}
print('🔎 Extração de informação com expressões regulares\n')
for nome, padrao in padroes.items():
    print(f'{nome:<13} {padrao:<32} → {re.findall(padrao, texto_rede_social)}')

print('\n⚠️ O que o nosso TokenizadorSimples faz com esse texto:')
print(TokenizadorSimples().tokenizar(texto_rede_social))

#### 🔎 Como interpretar o resultado

Cada padrão descreve a **forma** de uma informação:

| Padrão | Leitura | Detalhe importante |
|---|---|---|
| `(?<!\w)@\w+` | `@` seguido de letras/números/_ | `(?<!\w)` (*lookbehind* negativo) exige que **não** haja letra antes do `@`. Sem ele, o padrão capturaria `@exemplo` de dentro do e-mail |
| `#\w+` | `#` seguido de caracteres de palavra | `#IA2026`, `#PLN`, `#Python` |
| `https?://[^\s]+[^\s.,!?)]` | `http` ou `https`, `://` e tudo até um espaço | A última parte impede que o ponto final da frase entre na URL |
| `\d{2}/\d{2}/\d{4}` | 2 dígitos / 2 dígitos / 4 dígitos | Aceitaria `99/99/9999`: regex verifica **forma**, não **validade** |
| `R\$\s?\d+(?:\.\d{3})*,\d{2}` | `R$`, espaço opcional, número com milhar `.` e centavos `,` | Formato brasileiro de dinheiro |

- **O desastre do tokenizador simples (última linha):** a URL virou `httpsexemplocombrevento`, o e-mail virou `contatoexemplocombr`, a data virou `15032026` e o preço virou `r` e `4990`. Remover toda a pontuação **destrói** informação estruturada.
- **Ordem correta em um pipeline real:** primeiro **extrair/proteger** entidades com regex (URLs, e-mails, valores), depois tokenizar o restante. É exatamente o **Exercício 1** da lista.
- **💡 Conceito-chave:** regex são **regras escritas à mão**: precisas, rápidas e explicáveis, mas frágeis (um telefone escrito "11 98765 4321" não seria reconhecido). É a mesma troca entre regras e aprendizado que vimos nas Aulas 06–08.

### 2.7 Um corpus de verdade e a Lei de Zipf

Até aqui usamos frases inventadas. Agora vamos usar um **corpus** real: os **10 romances de Machado de Assis** (incluindo *Dom Casmurro* e *Memórias Póstumas de Brás Cubas*), em domínio público e distribuídos pelo NLTK.

Em 1949, **George Kingsley Zipf** observou que, em qualquer língua, a frequência de uma palavra é aproximadamente **inversamente proporcional** à sua posição no ranking:

$$\text{frequência}(r) \approx \frac{C}{r^{\alpha}}, \quad \alpha \approx 1$$

In [ ]:
# Corpus real: os 10 romances de Machado de Assis (domínio público), distribuídos pelo NLTK.
import zipfile
from collections import Counter

TEXTO_RESERVA = """Era uma vez uma cidade pequena onde todos se conheciam. A cidade tinha uma praça, uma igreja
e uma escola. Na escola, os alunos aprendiam a ler e a escrever, e o professor contava histórias sobre a cidade,
sobre a praça e sobre a igreja. Os alunos gostavam das histórias do professor e contavam as histórias aos pais.
""" * 40

def carregar_machado():
    try:
        import nltk
        nltk.download('machado', quiet=True)
        caminho = str(nltk.data.find('corpora/machado.zip'))
        with zipfile.ZipFile(caminho) as z:
            nomes = sorted(n for n in z.namelist() if n.startswith('machado/romance/') and n.endswith('.txt'))
            return '\n'.join(z.read(n).decode('latin-1') for n in nomes), len(nomes)
    except Exception as erro:
        print(f'⚠️ Não foi possível baixar o corpus ({erro.__class__.__name__}). Usando texto reserva (resultados menos expressivos).')
        return TEXTO_RESERVA, 0

corpus_machado, n_livros = carregar_machado()
palavras_machado = re.findall(r'[a-záéíóúâêôãõàçü]+', corpus_machado.lower())
freq_machado = Counter(palavras_machado)
print(f'Livros carregados: {n_livros} | caracteres: {len(corpus_machado):,} | '
      f'tokens: {len(palavras_machado):,} | vocabulário: {len(freq_machado):,} palavras distintas')
print('\nTrecho do início de Dom Casmurro:' if n_livros else '\nTrecho:')
i = corpus_machado.find('Uma noite destas')
print(corpus_machado[i:i + 230].replace('\n', ' ') + '...' if i >= 0 else corpus_machado[:230])

#### 🔎 Como interpretar o resultado

- **10 livros, ~3 milhões de caracteres e ~540 mil tokens:** um corpus pequeno para os padrões atuais (os LLMs são treinados com **trilhões** de tokens), mas grande o bastante para revelar regularidades estatísticas da língua.
- **Vocabulário de ~27 mil palavras distintas:** inclui todas as flexões (`olho`, `olhos`, `olhou`, `olhava`...), porque não aplicamos *stemming*.
- **Por que Machado de Assis?** Os textos estão em **domínio público**, têm boa qualidade e vêm prontos no NLTK. Além disso, um corpus literário do século XIX tem vocabulário e estilo **diferentes** do português atual da internet. Lembre disso ao interpretar os modelos treinados com ele mais adiante.
- Se aparecer o aviso de **texto reserva** (sem internet), os resultados das próximas células serão bem menos expressivos.

In [ ]:
ranking = freq_machado.most_common()
freqs = np.array([f for _, f in ranking])
ranks = np.arange(1, len(freqs) + 1)

print('As 15 palavras mais frequentes:')
for r, (p, f) in enumerate(ranking[:15], 1):
    print(f'  {r:2d}. {p:<6} {f:7,d}   rank × frequência = {r * f:7,d}')

total = freqs.sum()
cobertura_100 = freqs[:100].sum() / total
hapax = sum(1 for f in freqs if f == 1)
print(f'\nAs 100 palavras mais frequentes cobrem {cobertura_100:.0%} de todo o texto.')
print(f'{hapax:,} palavras ({hapax / len(freqs):.0%} do vocabulário) aparecem UMA única vez (hapax legomena).')

coef = np.polyfit(np.log(ranks[:5000]), np.log(freqs[:5000]), 1)
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
axs[0].bar([p for p, _ in ranking[:25]], freqs[:25], color=CORES['azul'])
axs[0].set_title('25 palavras mais frequentes'); axs[0].tick_params(axis='x', rotation=70)
axs[1].loglog(ranks, freqs, '.', ms=3, color=CORES['azul'], label='palavras do corpus')
axs[1].loglog(ranks, np.exp(coef[1]) * ranks ** coef[0], 'r-', label=f'reta ajustada: inclinação = {coef[0]:.2f}')
axs[1].set_xlabel('posição no ranking (log)'); axs[1].set_ylabel('frequência (log)')
axs[1].set_title('Lei de Zipf: frequência ∝ 1 / rank'); axs[1].legend()
plt.tight_layout(); plt.show()

#### 🔎 Como interpretar o resultado

- **As palavras mais frequentes são todas funcionais:** `a`, `que`, `de`, `e`, `o`, `não`, `se`... Só a palavra **`a`** aparece **23 mil** vezes. Repare em `lhe` (11ª), um traço do estilo de Machado que hoje seria raro.
- **"rank × frequência":** pela Lei de Zipf, deveria ser aproximadamente **constante**. No topo do ranking varia (23 mil a 77 mil), mas a partir da 6ª posição fica na faixa de 50 a 67 mil. A lei é uma **aproximação**, que vale melhor no meio da distribuição.
- **Gráfico log-log (direita):** os pontos formam quase uma **reta** com inclinação ≈ **−1,1**. Em escala log-log, uma lei de potência `f ∝ r^(−α)` vira uma reta de inclinação `−α`. Zipf previu α ≈ 1.
- **Dois números que mudam a forma de fazer PLN:**
  1. **100 palavras cobrem metade do texto.** Por isso remover stopwords reduz tanto o volume de dados, e por isso elas dominam qualquer contagem ingênua.
  2. **42% do vocabulário aparece uma única vez** (*hapax legomena*). Sempre haverá palavras **raras** ou **nunca vistas**, por maior que seja o corpus. É o problema da **esparsidade**, que motiva a suavização de probabilidades e a tokenização por subpalavras (próxima seção).

### 2.8 Tokenização por subpalavras: *Byte Pair Encoding* (BPE)

Tokenizar por palavras tem um problema: palavras que **nunca apareceram** no treino (*out-of-vocabulary*) ficam sem representação. Os modelos modernos (GPT, BERT, LLaMA, Claude) usam **subpalavras**: pedaços frequentes como `gat`, `inho`, `s`.

O **BPE** (Sennrich et al., 2016, adaptado de um algoritmo de compressão de 1994) aprende esses pedaços automaticamente:
1. Comece com cada palavra dividida em **caracteres**;
2. Conte todos os **pares de símbolos vizinhos** (ponderados pela frequência da palavra);
3. **Funda** o par mais frequente em um novo símbolo;
4. Repita até atingir o tamanho de vocabulário desejado.

In [ ]:
def treinar_bpe(freq_palavras, n_fusoes):
    # Byte Pair Encoding: começa com caracteres e funde, repetidamente, o PAR de símbolos mais frequente.
    vocab = {tuple(p) + ('</w>',): f for p, f in freq_palavras.items()}
    fusoes, historico = [], [dict(vocab)]
    for _ in range(n_fusoes):
        pares = Counter()
        for simbolos, f in vocab.items():
            for a, b in zip(simbolos, simbolos[1:]):
                pares[(a, b)] += f
        if not pares:
            break
        melhor = max(pares, key=pares.get)
        fusoes.append((melhor, pares[melhor]))
        novo = {}
        for simbolos, f in vocab.items():
            s, i = [], 0
            while i < len(simbolos):
                if i < len(simbolos) - 1 and (simbolos[i], simbolos[i + 1]) == melhor:
                    s.append(simbolos[i] + simbolos[i + 1]); i += 2
                else:
                    s.append(simbolos[i]); i += 1
            novo[tuple(s)] = f
        vocab = novo
        historico.append(dict(vocab))
    return fusoes, historico


def aplicar_bpe(palavra, fusoes):
    simbolos = list(palavra) + ['</w>']
    for (a, b), _ in fusoes:
        i, s = 0, []
        while i < len(simbolos):
            if i < len(simbolos) - 1 and simbolos[i] == a and simbolos[i + 1] == b:
                s.append(a + b); i += 2
            else:
                s.append(simbolos[i]); i += 1
        simbolos = s
    return simbolos


freq_bpe = {'casa': 10, 'casas': 6, 'casinha': 3, 'gato': 8, 'gatos': 5, 'gatinho': 3,
            'pato': 4, 'patos': 2, 'menina': 6, 'meninas': 3, 'menino': 5, 'meninos': 3}
fusoes_bpe, hist_bpe = treinar_bpe(freq_bpe, n_fusoes=14)
for k, ((a, b), n) in enumerate(fusoes_bpe, 1):
    print(f'fusão {k:2d}: "{a}" + "{b}" → "{a + b}"   (o par aparece {n} vezes)')

print('\nPalavras que NÃO estavam no treino, segmentadas em subpalavras conhecidas:')
for p in ['gatinhos', 'casinhas', 'patinho', 'meninada']:
    print(f'  {p:<10} → {aplicar_bpe(p, fusoes_bpe)}')

#### 🔎 Como interpretar o resultado

- **As primeiras fusões são os pares mais comuns**, ponderados pela frequência das palavras: `a+s` (28 = casa 10 + casas 2×6 + casinha 3 + meninas 3), `i+n` (23, em casinha, gatinho, menina, menino...) e `a+t` (22, em gato, gatos, gatinho, pato, patos). O algoritmo **não sabe nada de morfologia**: só conta pares.
- **Mesmo assim, ele descobre unidades linguisticamente plausíveis:**
  - `menin` (fusões 7–9): o **radical** comum a *menina, meninas, menino, meninos*;
  - `gat` e `cas`: radicais;
  - `os</w>` e `as</w>`: o **plural** no **fim** de palavra (`</w>` marca o fim);
  - `in` + `h`: o começo do **diminutivo** `-inho/-inha`.
- **O grande benefício (última parte da saída):** palavras que **nunca apareceram** no treino ainda são representáveis:
  - `gatinhos → gat + in + h + os` (radical + diminutivo + plural);
  - `meninada → menin + a + d + a`: o radical foi reconhecido, e o resto vira caracteres soltos.
  Nenhuma palavra fica "sem representação": no pior caso, ela vira uma sequência de caracteres.
- **💡 Ligação com os LLMs:** o GPT e outros modelos usam variantes do BPE com vocabulários de 50 a 200 mil subpalavras, aprendidos de trilhões de caracteres. É por isso que se fala em **tokens**, e não em palavras: uma palavra rara em português pode virar 3 ou 4 tokens.

In [ ]:
# 🎬 ANIMAÇÃO — o vocabulário de subpalavras sendo construído
cores_bpe = plt.cm.tab20(np.linspace(0, 1, 20))
fig, ax = plt.subplots(figsize=(11, 5.5))
palavras_bpe = list(freq_bpe)

def quadro_bpe(k):
    ax.clear(); ax.axis('off')
    vocab = hist_bpe[k]
    segs = {''.join(s).replace('</w>', ''): s for s in vocab}
    for i, p in enumerate(palavras_bpe):
        x = 0
        for simb in segs[p]:
            largura = 0.42 * len(simb.replace('</w>', '_'))
            cor = cores_bpe[sum(map(ord, simb)) % 20] if len(simb) > 1 else (0.92, 0.92, 0.92, 1)
            ax.add_patch(plt.Rectangle((x, -i), largura - 0.04, 0.8, color=cor, ec='k', lw=0.5))
            ax.text(x + largura / 2, -i + 0.4, simb.replace('</w>', '_'), ha='center', va='center', fontsize=10)
            x += largura
        ax.text(-0.3, -i + 0.4, f'{p} ({freq_bpe[p]})', ha='right', va='center', fontsize=10)
    ax.set_xlim(-3.5, 5.5); ax.set_ylim(-len(palavras_bpe) + 0.5, 1.4)
    if k == 0:
        titulo = 'Início: cada palavra é uma sequência de caracteres (_ = fim de palavra)'
    else:
        (a, b), n = fusoes_bpe[k - 1]
        titulo = f'Fusão {k}: "{a}" + "{b}" → "{a + b}"  (par mais frequente: {n} ocorrências)'
    ax.set_title(titulo, fontsize=11)

anim = animation.FuncAnimation(fig, quadro_bpe, frames=len(hist_bpe), interval=1100)
mostrar_animacao(anim)

#### 🔎 Como interpretar a animação

- Cada linha é uma palavra do corpus de treino (com sua frequência). Cada caixa é um **símbolo** do vocabulário atual. Caixas cinzas são caracteres isolados; caixas coloridas são **subpalavras** já fundidas (a mesma subpalavra tem sempre a mesma cor). O `_` marca o fim da palavra.
- **O que observar:**
  - A cada quadro, **um par** é fundido em **todas** as palavras em que aparece. Por exemplo, quando surge `menin`, as quatro palavras da família "menino" mudam juntas.
  - As palavras **mais frequentes** (como *casa*, 10 ocorrências) são "absorvidas" primeiro: no fim, `casa_` vira um **único** símbolo.
  - As **raras** (*casinha*, 3 ocorrências) continuam em pedaços: `cas` + `in` + `h` + `a_`.
- **💡 Conceito-chave:** BPE é um algoritmo de **compressão**. Palavras frequentes ganham símbolos próprios (custo baixo), raras são montadas com peças (sempre representáveis). É um equilíbrio entre o tamanho do vocabulário e o comprimento das sequências.

### 2.9 Distância de edição e correção ortográfica

Quantas operações de **inserção**, **remoção** ou **substituição** de um caractere transformam uma palavra em outra? Essa é a **distância de Levenshtein** (1965), calculada por **programação dinâmica**:

$$D[i, j] = \min \begin{cases} D[i-1, j] + 1 & \text{(remoção)} \\ D[i, j-1] + 1 & \text{(inserção)} \\ D[i-1, j-1] + [a_i \neq b_j] & \text{(substituição ou cópia)} \end{cases}$$

In [ ]:
def levenshtein(a, b, registrar=False):
    # Distância de edição: nº mínimo de inserções, remoções e substituições para transformar a em b.
    D = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    D[:, 0] = np.arange(len(a) + 1)
    D[0, :] = np.arange(len(b) + 1)
    passos = []
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            custo = 0 if a[i - 1] == b[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1,          # remoção
                          D[i, j - 1] + 1,          # inserção
                          D[i - 1, j - 1] + custo)  # substituição (ou cópia, se iguais)
        if registrar:
            passos.append(D.copy())
    return (D[-1, -1], passos) if registrar else D[-1, -1]


for a, b in [('gato', 'pato'), ('casa', 'casas'), ('amor', 'roma'), ('intenção', 'execução')]:
    print(f'distância("{a}", "{b}") = {levenshtein(a, b)}')

# 🎬 ANIMAÇÃO — preenchendo a tabela de programação dinâmica linha a linha
pa, pb = 'pesoa', 'pessoa'
dist, passos_lev = levenshtein(pa, pb, registrar=True)
fig, ax = plt.subplots(figsize=(7, 5.5))

def quadro_lev(k):
    ax.clear()
    D = passos_lev[k].astype(float)
    preenchidas = np.zeros_like(D, dtype=bool); preenchidas[:k + 2, :] = True; preenchidas[:, 0] = True
    ax.imshow(np.where(preenchidas, D, np.nan), cmap='YlGn_r', vmin=0, vmax=len(pb))
    for i in range(D.shape[0]):
        for j in range(D.shape[1]):
            if preenchidas[i, j]:
                ax.text(j, i, int(D[i, j]), ha='center', va='center', fontsize=12,
                        fontweight='bold' if (i, j) == (len(pa), len(pb)) else 'normal')
    ax.set_xticks(range(len(pb) + 1)); ax.set_xticklabels(['ε'] + list(pb), fontsize=12)
    ax.set_yticks(range(len(pa) + 1)); ax.set_yticklabels(['ε'] + list(pa), fontsize=12)
    ax.xaxis.tick_top()
    ax.set_title(f'Linha {k + 1} ("{pa[k]}") preenchida' + (f' → distância = {dist}' if k == len(pa) - 1 else ''), pad=25)

anim = animation.FuncAnimation(fig, quadro_lev, frames=len(passos_lev), interval=1200)
mostrar_animacao(anim)

#### 🔎 Como interpretar o resultado

- `gato → pato` = **1** (uma substituição) e `casa → casas` = **1** (uma inserção).
- `amor → roma` = **4**: mesmo sendo um **anagrama**, a distância é alta, porque Levenshtein compara **posições**, não conjuntos de letras.
- `intenção → execução` = **5**: é o exemplo clássico do livro de Jurafsky e Martin (*intention → execution*), adaptado ao português.

**Animação, a tabela de programação dinâmica:**
- A célula `D[i, j]` (linha `i`, coluna `j`) guarda a distância entre os **primeiros `i` caracteres** de `pesoa` e os **primeiros `j`** de `pessoa`. A linha e a coluna `ε` (cadeia vazia) são o caso base: transformar "" em `pes` custa 3 inserções.
- Cada célula é calculada a partir de **três vizinhas** (acima, à esquerda e na diagonal), escolhendo a operação mais barata. Por isso a tabela é preenchida **linha a linha**.
- **A diagonal verde-escura** (zeros) mostra os prefixos idênticos: `p = p`, `pe = pe`, `pes = pes`. Na linha do segundo `s` (que falta em `pesoa`), o custo sobe para 1, e a célula final vale **1**: basta **inserir um `s`**.
- **💡 Por que programação dinâmica?** Testar todas as sequências de edições seria exponencial. A tabela resolve cada **subproblema uma única vez** e custa `O(m × n)`, a mesma ideia do algoritmo de Viterbi e do alinhamento de sequências de DNA.

In [ ]:
# Um corretor ortográfico simples: vocabulário + frequências do corpus de Machado de Assis
vocab_corretor = {p: f for p, f in freq_machado.items() if f >= 3}

def distancia_edicao(a, b):
    # Mesma recorrência da função levenshtein, mas guardando só duas linhas (mais rápida).
    anterior = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        atual = [i]
        for j, cb in enumerate(b, 1):
            atual.append(min(anterior[j] + 1, atual[j - 1] + 1, anterior[j - 1] + (ca != cb)))
        anterior = atual
    return anterior[-1]

def corrigir(palavra, n=3):
    if palavra in vocab_corretor:
        return [(palavra, 0, vocab_corretor[palavra])]
    candidatos = [(c, distancia_edicao(palavra, c), f) for c, f in vocab_corretor.items() if abs(len(c) - len(palavra)) <= 2]
    candidatos.sort(key=lambda t: (t[1], -t[2]))     # menor distância; desempate pela palavra mais frequente
    return candidatos[:n]

print(f'Vocabulário do corretor: {len(vocab_corretor):,} palavras\n')
for erro in ['pesoa', 'cazamento', 'omem', 'filozofia', 'naum', 'coracao', 'olhoss']:
    sugestoes = corrigir(erro)
    print(f'{erro:<10} → ' + ', '.join(f'{c} (dist {d}, freq {f})' for c, d, f in sugestoes))

#### 🔎 Como interpretar o resultado

O corretor gera candidatos **do vocabulário** e os ordena por (1) **menor distância** e (2) **maior frequência** no corpus:

| Erro | 1ª sugestão | Por quê |
|---|---|---|
| `pesoa` | **pessoa** | Distância 1 empatada com `peso` e `pesa`. Vence a **mais frequente** (316 × 24 × 5) |
| `cazamento` | **casamento** | Troca clássica de `s` por `z` |
| `omem` | **homem** | O "h" mudo esquecido |
| `naum` | **num** ❌ | O usuário queria "não", mas `naum → não` exige 3 edições (`a→ã`, `u→o`, remover `m`) e `num` está a só 1 |
| `coracao` | **coração** ✅ | Distância 2: `c→ç` e `a→ã` contam como substituições comuns |
| `olhoss` | **olhos** | Remoção de uma letra repetida |

- **Por que `naum` falhou?** A distância de Levenshtein trata **todas** as edições como iguais. Mas trocar `a` por `ã` (esquecer o til) é um erro muito mais **provável** que trocar `a` por `x`. Corretores reais usam um **modelo de erro** com custos diferentes por operação (por exemplo, teclas vizinhas no teclado e acentos) e um **modelo de linguagem** que olha para as palavras ao redor ("eu **naum** sei" → "não" é muito mais provável no contexto).
- **💡 Conceito-chave:** essa combinação "modelo de erro × probabilidade da palavra" é o **modelo do canal ruidoso** (*noisy channel*). É a mesma estrutura usada em reconhecimento de fala e tradução automática estatística.

## 3. Representação Vetorial de Texto

Algoritmos de ML trabalham com números, não texto. Precisamos converter texto em vetores numéricos.

### 3.1 Bag-of-Words (Saco de Palavras)

**Bag-of-Words (BoW)** representa documento como vetor de contagens de palavras, ignorando ordem e gramática.

**Exemplo:**
- Doc1: "gato come peixe"
- Doc2: "cachorro come carne"

Vocabulário: [gato, cachorro, come, peixe, carne]

- Vec1: [1, 0, 1, 1, 0]
- Vec2: [0, 1, 1, 0, 1]

**Vantagens:** Simples, eficaz para classificação de textos.

**Desvantagens:** Perde ordem, não captura semântica, alta dimensionalidade, esparso.

### 3.2 Implementação de Bag-of-Words

In [ ]:
import numpy as np
from typing import List, Dict

class BagOfWords:
    """
    Implementação de Bag-of-Words.
    """
    
    def __init__(self):
        self.vocabulario: Dict[str, int] = {}
        self.tokenizador = TokenizadorSimples()
    
    def fit(self, documentos: List[str]):
        """
        Constrói vocabulário a partir dos documentos.
        """
        todas_palavras = set()
        
        for doc in documentos:
            tokens = self.tokenizador.tokenizar(doc)
            tokens = self.tokenizador.remover_stopwords(tokens)
            todas_palavras.update(tokens)
        
        # Cria mapeamento palavra -> índice
        self.vocabulario = {palavra: idx for idx, palavra in enumerate(sorted(todas_palavras))}
        
        print(f"Vocabulário construído: {len(self.vocabulario)} palavras únicas")
    
    def transform(self, documentos: List[str]) -> np.ndarray:
        """
        Transforma documentos em vetores BoW.
        """
        n_docs = len(documentos)
        n_features = len(self.vocabulario)
        
        # Matriz de contagens
        matriz = np.zeros((n_docs, n_features), dtype=int)
        
        for i, doc in enumerate(documentos):
            tokens = self.tokenizador.tokenizar(doc)
            tokens = self.tokenizador.remover_stopwords(tokens)
            
            for token in tokens:
                if token in self.vocabulario:
                    idx = self.vocabulario[token]
                    matriz[i, idx] += 1
        
        return matriz
    
    def fit_transform(self, documentos: List[str]) -> np.ndarray:
        """
        Constrói vocabulário e transforma documentos.
        """
        self.fit(documentos)
        return self.transform(documentos)

# Exemplo
print("="*60)
print("BAG-OF-WORDS (SACO DE PALAVRAS)")
print("="*60)

documentos = [
    "O gato subiu no telhado",
    "O cachorro late muito alto",
    "O gato e o cachorro são animais de estimação",
    "O pássaro voa alto no céu"
]

print("\nDocumentos:")
for i, doc in enumerate(documentos):
    print(f"  Doc{i+1}: {doc}")

bow = BagOfWords()
matriz_bow = bow.fit_transform(documentos)

print(f"\n\nMatriz BoW shape: {matriz_bow.shape}")
print(f"(Documentos x Vocabulário)\n")

# Mostra vocabulário
print("Vocabulário (primeiras 10 palavras):")
palavras = sorted(bow.vocabulario.keys())
for i, palavra in enumerate(palavras[:10]):
    print(f"  {i}: {palavra}")

# Mostra matriz
print(f"\n\nMatriz BoW (primeiras 5 colunas):")
print(matriz_bow[:, :5])

# Similaridade entre documentos
from numpy.linalg import norm

def similaridade_cosseno(v1, v2):
    """Calcula similaridade de cosseno entre dois vetores."""
    if norm(v1) == 0 or norm(v2) == 0:
        return 0
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

print("\n\nSimilaridade entre documentos (cosseno):")
print(f"  Doc1 vs Doc3: {similaridade_cosseno(matriz_bow[0], matriz_bow[2]):.3f}")
print(f"  Doc2 vs Doc3: {similaridade_cosseno(matriz_bow[1], matriz_bow[2]):.3f}")

print("\n" + "="*60)

#### 🔎 Como interpretar o resultado

- **Vocabulário de 11 palavras:** as stopwords (`o`, `no`, `e`, `muito`, `são`, `de`) foram removidas antes de montar o vocabulário.
- **Matriz 4 × 11:** cada **linha** é um documento e cada **coluna** uma palavra. A célula indica **quantas vezes** a palavra aparece no documento.
- **Por que a 1ª linha só tem zeros nas 5 primeiras colunas?** O Doc1 ("O gato subiu no telhado") contém `gato`, `subiu` e `telhado`, que estão nas colunas 5, 8 e 9, fora da parte impressa. Isso mostra a **esparsidade**: a maioria das células é zero.
- **Similaridades:**
  - Doc1 × Doc3 = **0,289**: compartilham só `gato`. Pelo cosseno: 1 / √(3 × 4) = 0,289, porque Doc1 tem 3 palavras e Doc3 tem 4;
  - Doc2 × Doc3 = **0,289**: compartilham só `cachorro`, com a mesma conta.
- **Leitura semântica:** para o BoW, "gato" e "cachorro" são tão diferentes quanto "gato" e "céu". Ele só reconhece **palavras idênticas**, não palavras de significado próximo. Os *embeddings* (Seção 4.6) resolvem isso.

**Observações:**

1. **Esparsidade:** A maioria das entradas é zero — documentos usam pequena fração do vocabulário.

2. **Similaridade de cosseno:** Mede ângulo entre vetores, ignorando magnitude. Valores próximos a 1 indicam alta similaridade.

3. **Limitações:** "O gato persegue o rato" e "O rato persegue o gato" têm mesma representação BoW, mas significados opostos.

### 3.2.1 Visualizando a matriz BoW e as similaridades

In [ ]:
vocab_ordenado = sorted(bow.vocabulario, key=bow.vocabulario.get)
sim = np.array([[similaridade_cosseno(a, b) for b in matriz_bow] for a in matriz_bow])

fig, axs = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={'width_ratios': [2.2, 1]})
axs[0].imshow(matriz_bow, cmap='Blues', aspect='auto')
axs[0].set_xticks(range(len(vocab_ordenado))); axs[0].set_xticklabels(vocab_ordenado, rotation=45, ha='right')
axs[0].set_yticks(range(4)); axs[0].set_yticklabels([f'Doc{i + 1}' for i in range(4)])
for i in range(matriz_bow.shape[0]):
    for j in range(matriz_bow.shape[1]):
        axs[0].text(j, i, matriz_bow[i, j], ha='center', va='center', color='w' if matriz_bow[i, j] else '#ccc')
axs[0].set_title('Matriz documento × termo (Bag-of-Words)')
im = axs[1].imshow(sim, cmap='Greens', vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        axs[1].text(j, i, f'{sim[i, j]:.2f}', ha='center', va='center')
axs[1].set_xticks(range(4)); axs[1].set_xticklabels([f'D{i + 1}' for i in range(4)])
axs[1].set_yticks(range(4)); axs[1].set_yticklabels([f'D{i + 1}' for i in range(4)])
axs[1].set_title('Similaridade de cosseno')
plt.tight_layout(); plt.show()

#### 🔎 Como interpretar os gráficos

- **Esquerda, matriz documento × termo:** quadrados escuros indicam que a palavra aparece no documento. Repare que **cada coluna tem poucos quadrados escuros**. Em coleções reais, com vocabulários de 100 mil palavras, mais de 99% das células são zero, e por isso essas matrizes são guardadas em formato **esparso**.
- **Direita, similaridade de cosseno entre documentos:**
  - **diagonal = 1,00**: todo documento é idêntico a si mesmo;
  - **Doc1 × Doc2 = 0,00**: não têm nenhuma palavra em comum (ortogonais);
  - **Doc2 × Doc4 = 0,29**: ambos têm `alto` (o cachorro late **alto** e o pássaro voa **alto**). O BoW considera os documentos parecidos por causa de uma palavra usada em **sentidos diferentes** (polissemia).
- **💡 Conceito-chave:** o cosseno mede o **ângulo** entre os vetores, não o tamanho. Um documento longo e um curto sobre o mesmo assunto podem ter cosseno alto.

### 3.2.2 N-gramas: recuperando um pouco da ordem

O BoW ignora a ordem: "o gato persegue o rato" e "o rato persegue o gato" viram o **mesmo vetor**. Uma solução simples é contar também **sequências de n palavras** (*n-gramas*): bigramas (n = 2), trigramas (n = 3)...

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

frases = ['o gato persegue o rato', 'o rato persegue o gato']
for n in [(1, 1), (1, 2)]:
    vet = CountVectorizer(ngram_range=n)
    M = vet.fit_transform(frases).toarray()
    cos = similaridade_cosseno(M[0], M[1])
    print(f'n-gramas {n}: vocabulário = {list(vet.get_feature_names_out())}')
    print(f'   frase 1 → {M[0]}')
    print(f'   frase 2 → {M[1]}')
    print(f'   similaridade de cosseno = {cos:.3f}\n')

#### 🔎 Como interpretar o resultado

- **Só unigramas:** as duas frases viram exatamente `[1, 1, 1]`, com **similaridade 1,000**. Para o BoW, "o gato persegue o rato" e "o rato persegue o gato" **são a mesma frase**, embora o sentido seja oposto.
- **Unigramas + bigramas:** surgem as colunas `gato persegue`, `persegue rato`, `rato persegue` e `persegue gato`. Agora as frases diferem e a **similaridade cai para 0,600**: elas ainda compartilham as palavras, mas não a **ordem**.
- **Por que não usar sempre n-gramas grandes?** O vocabulário **explode**: com 10 mil palavras, há até 100 milhões de bigramas possíveis, e a maioria aparece uma vez só (esparsidade, Lei de Zipf). Na prática, usa-se n = 1 a 3.
- **💡 Conceito-chave:** n-gramas capturam **ordem local** ("não gostei", "machine learning", "São Paulo"). É uma solução barata e muito eficaz, usada em classificadores e nos modelos de linguagem da Seção 4.5.

### 3.3 TF-IDF (Term Frequency - Inverse Document Frequency)

**TF-IDF** pondera palavras pela importância no documento e raridade no corpus.

**Fórmula:**

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Onde:

**Term Frequency (TF):**
$$\text{TF}(t, d) = \frac{\text{frequência de } t \text{ em } d}{\text{total de termos em } d}$$

**Inverse Document Frequency (IDF):**
$$\text{IDF}(t) = \log\left(\frac{\text{total de documentos}}{\text{documentos contendo } t}\right)$$

**Intuição:**
- Palavras frequentes no documento mas raras no corpus recebem peso alto
- Palavras em todos os documentos (como stopwords) recebem peso baixo

### 3.4 Implementação de TF-IDF

In [ ]:
import numpy as np
from collections import Counter
import math

class TFIDF:
    """
    Implementação de TF-IDF.
    """
    
    def __init__(self):
        self.vocabulario: Dict[str, int] = {}
        self.idf: np.ndarray = None
        self.tokenizador = TokenizadorSimples()
    
    def fit(self, documentos: List[str]):
        """
        Calcula IDF para cada termo no vocabulário.
        """
        # Constrói vocabulário
        todas_palavras = set()
        for doc in documentos:
            tokens = self.tokenizador.tokenizar(doc)
            tokens = self.tokenizador.remover_stopwords(tokens)
            todas_palavras.update(tokens)
        
        self.vocabulario = {palavra: idx for idx, palavra in enumerate(sorted(todas_palavras))}
        
        # Calcula IDF
        n_docs = len(documentos)
        n_features = len(self.vocabulario)
        
        # Conta em quantos documentos cada termo aparece
        df = np.zeros(n_features)  # document frequency
        
        for doc in documentos:
            tokens = self.tokenizador.tokenizar(doc)
            tokens = self.tokenizador.remover_stopwords(tokens)
            tokens_unicos = set(tokens)
            
            for token in tokens_unicos:
                if token in self.vocabulario:
                    idx = self.vocabulario[token]
                    df[idx] += 1
        
        # IDF = log(N / df)
        # Adiciona 1 ao denominador para evitar divisão por zero
        self.idf = np.log((n_docs + 1) / (df + 1)) + 1
        
        print(f"Vocabulário: {len(self.vocabulario)} termos")
    
    def transform(self, documentos: List[str]) -> np.ndarray:
        """
        Transforma documentos em vetores TF-IDF.
        """
        n_docs = len(documentos)
        n_features = len(self.vocabulario)
        
        # Matriz TF
        tf_matriz = np.zeros((n_docs, n_features))
        
        for i, doc in enumerate(documentos):
            tokens = self.tokenizador.tokenizar(doc)
            tokens = self.tokenizador.remover_stopwords(tokens)
            
            if len(tokens) == 0:
                continue
            
            # Conta frequência de cada termo
            freq = Counter(tokens)
            
            for termo, count in freq.items():
                if termo in self.vocabulario:
                    idx = self.vocabulario[termo]
                    # TF normalizado
                    tf_matriz[i, idx] = count / len(tokens)
        
        # TF-IDF = TF * IDF
        tfidf_matriz = tf_matriz * self.idf
        
        return tfidf_matriz
    
    def fit_transform(self, documentos: List[str]) -> np.ndarray:
        self.fit(documentos)
        return self.transform(documentos)
    
    def termos_mais_importantes(self, documento: str, top_n=5):
        """
        Retorna termos mais importantes (maior TF-IDF) de um documento.
        """
        vetor = self.transform([documento])[0]
        
        # Obtém índices ordenados por valor TF-IDF
        indices_ordenados = np.argsort(vetor)[::-1][:top_n]
        
        # Mapeia índices de volta para palavras
        idx_para_palavra = {idx: palavra for palavra, idx in self.vocabulario.items()}
        
        return [(idx_para_palavra[idx], vetor[idx]) for idx in indices_ordenados if vetor[idx] > 0]

# Exemplo
print("="*60)
print("TF-IDF (TERM FREQUENCY - INVERSE DOCUMENT FREQUENCY)")
print("="*60)

docs_exemplo = [
    "Python é uma linguagem de programação muito popular",
    "Machine learning utiliza algoritmos de aprendizado",
    "Deep learning é uma subárea de machine learning",
    "Python é usado para machine learning e data science"
]

print("\nDocumentos:")
for i, doc in enumerate(docs_exemplo):
    print(f"  Doc{i+1}: {doc}")

tfidf = TFIDF()
matriz_tfidf = tfidf.fit_transform(docs_exemplo)

print(f"\n\nMatriz TF-IDF shape: {matriz_tfidf.shape}")

# Mostra termos mais importantes de cada documento
print("\nTermos mais importantes por documento:\n")
for i, doc in enumerate(docs_exemplo):
    print(f"Doc{i+1}:")
    termos = tfidf.termos_mais_importantes(doc, top_n=3)
    for termo, score in termos:
        print(f"  {termo}: {score:.3f}")
    print()

# Similaridade
print("\nSimilaridade entre documentos (TF-IDF):")
print(f"  Doc1 vs Doc4: {similaridade_cosseno(matriz_tfidf[0], matriz_tfidf[3]):.3f}")
print(f"  Doc2 vs Doc3: {similaridade_cosseno(matriz_tfidf[1], matriz_tfidf[2]):.3f}")

print("\n" + "="*60)

#### 🔎 Como interpretar o resultado

| Documento | Termos de maior peso | Por quê |
|---|---|---|
| Doc1 | `programação`, `popular`, `linguagem` (0,479) | Exclusivos do Doc1 (IDF alto). `python` também está no Doc4, por isso pesa menos |
| Doc2 | `utiliza`, `aprendizado`, `algoritmos` (0,383) | Exclusivos, mas o Doc2 é mais longo, então o TF de cada termo é menor |
| Doc3 | **`learning` (0,489)** | Aparece em 3 documentos (IDF baixo), mas **duas vezes** no Doc3 (TF alto) |
| Doc4 | `usado`, `science`, `data` (0,319) | Exclusivos, mas o Doc4 tem 6 termos de conteúdo (TF = 1/6) |

- **Conta para `learning` no Doc3:** TF = 2/5 = 0,4; IDF = ln(5/4) + 1 ≈ 1,22; TF-IDF ≈ **0,489**. Para `deep`: TF = 1/5, IDF = ln(5/2) + 1 ≈ 1,92, então **0,383**.
- **Similaridades:** Doc1 × Doc4 = **0,155** (só compartilham `python`); Doc2 × Doc3 = **0,311** (compartilham `machine` e `learning`). Como esses termos são **comuns** no corpus, o IDF reduz o peso deles e a similaridade fica moderada.
- **💡 Conceito-chave:** TF-IDF responde *"que palavras caracterizam ESTE documento em relação aos outros?"*. Uma palavra que está em todo lugar não ajuda a distinguir nada.

**Observações:**

1. **IDF penaliza termos comuns:** Palavras que aparecem em muitos documentos ("machine", "learning") recebem IDF menor.

2. **TF captura importância local:** Palavras frequentes no documento recebem TF alto.

3. **Normalização:** TF é normalizado pelo comprimento do documento para evitar viés para documentos longos.

4. **Aplicações:** TF-IDF é amplamente usado em recuperação de informação, busca, classificação de textos e extração de palavras-chave.

### 3.4.1 TF-IDF calculado à mão (passo a passo)

Antes de confiar na implementação, vale fazer a conta com um exemplo pequeno, como faríamos no quadro. Três documentos:

- **D1:** "gato come peixe"
- **D2:** "gato dorme"
- **D3:** "cachorro come carne carne"

**Passo 1, IDF** (com $N = 3$ documentos e logaritmo natural):

| termo | em quantos docs (df) | IDF = ln(N/df) |
|---|:-:|:-:|
| gato | 2 | ln(3/2) = **0,405** |
| come | 2 | ln(3/2) = **0,405** |
| peixe, dorme, cachorro, carne | 1 | ln(3/1) = **1,099** |

**Passo 2, TF** de "carne" em D3: aparece 2 vezes em 4 palavras → TF = 2/4 = **0,5**.

**Passo 3, TF-IDF** de "carne" em D3: 0,5 × 1,099 = **0,549**. Já "come" em D3: (1/4) × 0,405 = **0,101**. *carne* é ~5× mais importante que *come* para descrever D3: é **frequente no documento** e **rara na coleção**.

A célula abaixo refaz a conta e compara com o `TfidfVectorizer` do scikit-learn.

In [ ]:
# TF-IDF "de quadro": a fórmula clássica, passo a passo, e a versão do scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
docs_mao = ['gato come peixe', 'gato dorme', 'cachorro come carne carne']
toks_mao = [d.split() for d in docs_mao]
vocab_mao = sorted({w for t in toks_mao for w in t})
N = len(docs_mao)

df = {w: sum(w in t for t in toks_mao) for w in vocab_mao}
idf_classico = {w: math.log(N / df[w]) for w in vocab_mao}

print(f'{"termo":<10}{"df":>3}{"IDF":>8}   ' + '   '.join(f'TF-IDF D{i+1}' for i in range(N)))
for w in vocab_mao:
    tfidf = [t.count(w) / len(t) * idf_classico[w] for t in toks_mao]
    print(f'{w:<10}{df[w]:>3}{idf_classico[w]:>8.3f}   ' + '   '.join(f'{v:>10.3f}' for v in tfidf))

# scikit-learn usa por padrão: TF = contagem bruta, IDF suavizado = ln((1+N)/(1+df)) + 1, e normaliza cada linha (norma L2)
sk = TfidfVectorizer().fit(docs_mao)
print('\nIDF do scikit-learn:', {w: round(v, 3) for w, v in zip(sk.get_feature_names_out(), sk.idf_)})
print('Vetor TF-IDF (sklearn) de D3:', dict(zip(sk.get_feature_names_out(), sk.transform([docs_mao[2]]).toarray()[0].round(3))))

#### 🔎 Como interpretar o resultado

- **A tabela confere com a conta do quadro:** `carne` em D3 = **0,549** e `come` em D3 = **0,101**. Repare que `cachorro` e `carne` têm o mesmo IDF, mas `carne` vale o dobro (0,549 × 0,275) porque aparece duas vezes: é o **TF** fazendo diferença.
- **Termos que aparecem em 2 dos 3 documentos** (`gato`, `come`) têm IDF baixo (0,405); os que aparecem em **um só** têm IDF alto (1,099). Se uma palavra estivesse nos 3 documentos, o IDF seria ln(3/3) = **0**: ela seria inútil para diferenciar documentos (é o que acontece com stopwords).
- **O scikit-learn dá números diferentes**, e isso confunde muita gente. Ele usa:
  1. **IDF suavizado**, ln((1+N)/(1+df)) + 1: o "+1" dentro do log evita divisão por zero; o "+1" fora garante que nenhum termo tenha peso **zero**;
  2. **TF bruto** (contagem), em vez de dividir pelo tamanho do documento;
  3. **Normalização L2** de cada vetor (norma 1), que deixa documentos longos e curtos comparáveis pelo cosseno.

  A **ordem** de importância é preservada: em D3, `carne` (0,847) continua muito acima de `come` (0,322) e de `cachorro` (0,423). A classe `TFIDF` da Seção 3.4 usa o mesmo IDF suavizado do scikit-learn.
- **💡 Conceito-chave:** "TF-IDF" é uma **família** de fórmulas, não uma fórmula única. Ao comparar resultados entre bibliotecas, verifique sempre qual variante de TF, IDF e normalização está sendo usada.

### 3.5 🎛️ Um mini buscador com TF-IDF

Um mecanismo de busca clássico (o **modelo vetorial** de Gerard Salton, anos 1970) faz exatamente isto: representa documentos e consulta como vetores TF-IDF e ordena os documentos pela **similaridade de cosseno** com a consulta. Agora usamos o `TfidfVectorizer` do scikit-learn, versão otimizada do que implementamos acima.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

colecao = [
    'Receita de bolo de chocolate com cobertura de brigadeiro',
    'Como treinar uma rede neural para reconhecer imagens de gatos',
    'Dicas para cuidar de gatos filhotes em apartamento',
    'Campeonato brasileiro de futebol: resultados da rodada',
    'Aprendizado de máquina na previsão do tempo e de chuvas',
    'Receita de pão caseiro sem sova para iniciantes',
    'O time venceu o campeonato com gol no último minuto',
    'Previsão do tempo: frente fria traz chuva para o Sul',
    'Python para ciência de dados: pandas, numpy e aprendizado de máquina',
    'Gatos e cachorros: como apresentar os animais em casa',
    'Bolo de cenoura com chocolate: receita da vovó',
    'Redes neurais profundas e o futuro da inteligência artificial',
]
stop_pt = sorted(TokenizadorSimples().stopwords)
motor = TfidfVectorizer(stop_words=stop_pt)
M_colecao = motor.fit_transform(colecao)

idf = dict(zip(motor.get_feature_names_out(), motor.idf_))
print('Termos COMUNS (IDF baixo):', sorted(idf, key=idf.get)[:6])
print('Termos RAROS  (IDF alto) :', sorted(idf, key=idf.get, reverse=True)[:6], '\n')

def buscar(consulta='receita de chocolate', top=5):
    q = motor.transform([consulta])
    scores = (M_colecao @ q.T).toarray().ravel()          # vetores já normalizados → produto = cosseno
    ordem = np.argsort(scores)[::-1][:top]
    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.barh([colecao[i][:55] for i in ordem][::-1], scores[ordem][::-1], color=CORES['azul'])
    ax.set_xlim(0, 1); ax.set_title(f'Resultados para: "{consulta}"  (similaridade de cosseno TF-IDF)')
    plt.tight_layout(); plt.show()
    termos = [t for t in motor.build_analyzer()(consulta) if t in idf]
    print('Termos da consulta reconhecidos e seus IDFs:', {t: round(idf[t], 2) for t in termos})

interact(buscar, consulta=widgets.Text(value='receita de chocolate', description='Busca:', continuous_update=False),
         top=widgets.fixed(5));

#### 🔎 Como usar e interpretar

- **Termos comuns × raros:** `receita`, `gatos`, `bolo`, `chocolate` aparecem em 2–3 dos 12 documentos (IDF baixo); `brigadeiro` e `apartamento` aparecem em um só (IDF alto). Um termo raro na consulta é muito mais **discriminante**.
- **Consulta padrão, "receita de chocolate":**
  - os dois bolos de chocolate empatam no topo (≈ 0,57): têm **as duas** palavras da consulta;
  - o pão caseiro aparece em 3º (≈ 0,23): tem `receita`, mas não `chocolate`;
  - os demais têm similaridade 0: nenhuma palavra em comum.
  - `de` foi ignorado: é uma stopword.

**Experimente:**
1. `gatos` → aparecem os **três** documentos que contêm a palavra, inclusive o de **redes neurais** que reconhecem imagens de gatos. A busca não sabe do que o documento **trata**, só quais palavras ele contém;
2. `chuva` → só a previsão do tempo... mas `chuvas` (plural) acha **outro** documento. Sem *stemming*, plural e singular são palavras diferentes;
3. `inteligência artificial` × `IA` → a sigla não é encontrada: TF-IDF não entende **sinônimos**;
4. `futebol` → o campeonato brasileiro aparece, mas "O time venceu o campeonato" **não**, embora seja sobre futebol.

- **💡 Conceito-chave:** é uma **busca lexical**: casa **palavras**, não **significados**. Buscadores modernos combinam esse modelo (e o seu sucessor, o BM25) com **busca semântica** por *embeddings*, que encontraria "time venceu o campeonato" para a consulta "futebol".

## 4. Análise de Sentimentos

**Análise de Sentimentos** (Sentiment Analysis) identifica e extrai opiniões subjetivas de texto, classificando como positivo, negativo ou neutro.

### 4.1 Abordagens

#### Baseada em Léxico
Usa dicionários de palavras com polaridades ("excelente" = +3, "ruim" = -2).

#### Baseada em Machine Learning
Treina classificador (Naive Bayes, SVM, etc.) com textos rotulados.

#### Deep Learning
Redes neurais (LSTM, Transformers) aprendem representações complexas.

### 4.2 Desafios

- **Sarcasmo:** "Que produto maravilhoso!" (negativo)
- **Negação:** "Não é ruim" (positivo)
- **Contexto:** "O filme é longo" — positivo ou negativo?
- **Multilingual:** Expressões idiomáticas variam por idioma

### 4.3 Implementação Simples de Análise de Sentimentos

In [ ]:
class AnalisadorSentimentos:
    """
    Analisador de sentimentos baseado em léxico para português.
    """
    
    def __init__(self):
        # Léxico simplificado de sentimentos
        self.palavras_positivas = {
            'excelente': 3, 'ótimo': 3, 'maravilhoso': 3, 'perfeito': 3,
            'bom': 2, 'legal': 2, 'gostei': 2, 'adorei': 3, 'incrível': 3,
            'feliz': 2, 'alegre': 2, 'satisfeito': 2, 'sucesso': 2,
            'qualidade': 1, 'eficiente': 2, 'rápido': 1, 'fácil': 1
        }
        
        self.palavras_negativas = {
            'péssimo': -3, 'horrível': -3, 'terrível': -3, 'ruim': -2,
            'mal': -2, 'problema': -1, 'defeito': -2, 'falha': -2,
            'lento': -1, 'difícil': -1, 'caro': -1, 'frustrado': -2,
            'decepcionado': -2, 'insatisfeito': -2, 'chato': -1
        }
        
        # Modificadores
        self.intensificadores = {'muito', 'extremamente', 'super', 'demais'}
        self.negacoes = {'não', 'nem', 'nunca', 'jamais', 'nada'}
        
        self.tokenizador = TokenizadorSimples()
    
    def calcular_sentimento(self, texto: str) -> Dict:
        """
        Calcula score de sentimento do texto.
        
        Retorna:
        --------
        Dict com score, polaridade e palavras detectadas.
        """
        tokens = self.tokenizador.tokenizar(texto)
        
        score = 0
        palavras_detectadas = []
        
        i = 0
        while i < len(tokens):
            token = tokens[i]
            
            # Verifica negação em uma pequena janela antes da palavra.
            # Esta é uma heurística simples: procura palavras de negação
            # até alguns tokens antes, interrompendo em conjunções adversativas
            # (por exemplo, "mas"). Ainda assim, não cobre todos os casos.
            negacao = False
            if i > 0:
                janela_negacao = 3  # número máximo de tokens para olhar para trás
                conj_ruptura = {"mas", "porém", "contudo", "entretanto"}
                inicio = max(0, i - janela_negacao)
                for j in range(i - 1, inicio - 1, -1):
                    # Se encontramos uma conjunção adversativa, paramos:
                    # a negação antes dela não deve afetar o token atual.
                    if tokens[j] in conj_ruptura:
                        break
                    if tokens[j] in self.negacoes:
                        negacao = True
                        break
            
            # Verifica intensificador (ainda apenas imediatamente antes)
            intensificador = False
            if i > 0 and tokens[i-1] in self.intensificadores:
                intensificador = True
            
            # Calcula score
            palavra_score = 0
            if token in self.palavras_positivas:
                palavra_score = self.palavras_positivas[token]
                palavras_detectadas.append((token, 'positiva'))
            elif token in self.palavras_negativas:
                palavra_score = self.palavras_negativas[token]
                palavras_detectadas.append((token, 'negativa'))
            
            # Aplica modificadores
            if palavra_score != 0:
                if negacao:
                    palavra_score *= -0.5  # Inverte e reduz
                if intensificador:
                    palavra_score *= 1.5
                
                score += palavra_score
            
            i += 1
        
        # Determina polaridade
        if score > 0.5:
            polaridade = 'POSITIVO'
        elif score < -0.5:
            polaridade = 'NEGATIVO'
        else:
            polaridade = 'NEUTRO'
        
        return {
            'score': score,
            'polaridade': polaridade,
            'palavras': palavras_detectadas
        }

# Exemplo
print("="*60)
print("ANÁLISE DE SENTIMENTOS")
print("="*60)

analisador = AnalisadorSentimentos()

textos_teste = [
    "Este produto é excelente! Estou muito satisfeito.",
    "Péssima qualidade. Muito decepcionado com a compra.",
    "O atendimento foi bom, mas o produto não é tão legal.",
    "Não gostei do filme. Achei muito chato.",
    "O restaurante não é ruim, a comida é boa."
]

print("\nAnálise de textos:\n")
for texto in textos_teste:
    resultado = analisador.calcular_sentimento(texto)
    
    print(f"Texto: \"{texto}\"")
    print(f"  Score: {resultado['score']:.2f}")
    print(f"  Polaridade: {resultado['polaridade']}")
    if resultado['palavras']:
        print(f"  Palavras detectadas: {resultado['palavras']}")
    print()

print("="*60)

#### 🔎 Como interpretar o resultado

| Texto | Score | Como o score foi calculado | Correto? |
|---|:-:|---|:-:|
| "excelente! ... muito satisfeito" | **+6,0** | excelente (+3) + muito × satisfeito (2 × 1,5 = +3) | ✅ |
| "Péssima qualidade. Muito decepcionado" | **−2,0** | `péssima` **não foi detectada** (o léxico só tem `péssimo`); `qualidade` contou **+1**; muito × decepcionado = −3 | ✅ por sorte |
| "bom, mas ... não é tão legal" | **+1,0** | bom (+2) + legal negado (+2 × −0,5 = −1) | ✅ (misto) |
| "Não gostei ... muito chato" | **−2,5** | gostei negado (−1) + muito × chato (−1,5) | ✅ |
| "não é ruim, a comida é boa" | **+1,0** | ruim negado (−2 × −0,5 = +1); `boa` **não detectada** (só `bom` no léxico) | ✅ por sorte |

- **Os acertos escondem fragilidades:** duas frases acertaram **por acaso**. O léxico não reconhece o **feminino** (`péssima`, `boa`) e trata `qualidade` como positiva mesmo em "péssima qualidade". Lematizar antes (`péssima → péssimo`) resolveria o primeiro problema.
- **Por que a negação multiplica por −0,5 e não por −1?** "Não é ruim" **não** significa "é ótimo". A negação inverte e **atenua** o sentimento, uma heurística observada empiricamente.
- **💡 Conceito-chave:** abordagens por léxico são **transparentes** (dá para explicar cada ponto do score) e não precisam de dados de treino, mas exigem muito trabalho manual para cobrir a variação da língua. Na próxima seção comparamos com um modelo **aprendido**.

**Observações:**

1. **Léxico limitado:** Implementação usa vocabulário reduzido. Na prática, use léxicos completos como OpLexicon ou SentiLex para português.

2. **Negação:** "Não gostei" inverte polaridade. A implementação procura palavras de negação em uma janela de até 3 palavras anteriores, interrompida por conjunções adversativas ("mas", "porém").

3. **Intensificadores:** "Muito bom" amplifica sentimento positivo.

4. **Limitações:** Não captura sarcasmo, contexto complexo ou sentimentos mistos adequadamente. Para precisão maior, use modelos treinados com ML/Deep Learning.

### 4.4 Aprendendo o sentimento a partir de dados (Naive Bayes)

Em vez de escrever um léxico à mão, podemos **aprender** quais palavras indicam sentimento a partir de exemplos rotulados (Aula 06). Montamos o experimento como em um projeto real, com **dados de treino e de teste de origens diferentes**:

- **Treino:** 200 avaliações geradas automaticamente a partir de modelos de frase ("Produto: excelente. Recomendo!");
- **Teste:** 40 avaliações **escritas à mão**, com vocabulário natural e variado (20 positivas e 20 negativas).

O classificador é um **Naive Bayes** sobre TF-IDF de unigramas e bigramas.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# TESTE: 40 avaliações escritas à mão, com vocabulário variado e natural
avaliacoes = [
    ('Produto excelente, chegou rápido e funciona perfeitamente', 1),
    ('Adorei a compra, qualidade muito boa', 1),
    ('Ótimo custo-benefício, recomendo a todos', 1),
    ('Atendimento nota dez, resolveram meu problema na hora', 1),
    ('Superou minhas expectativas, estou muito feliz', 1),
    ('A comida estava deliciosa e o ambiente agradável', 1),
    ('Filme emocionante, roteiro incrível e ótimas atuações', 1),
    ('Entrega antes do prazo e embalagem caprichada', 1),
    ('Gostei bastante, voltarei a comprar com certeza', 1),
    ('O hotel é confortável, limpo e bem localizado', 1),
    ('Aplicativo fácil de usar e muito útil no dia a dia', 1),
    ('Livro maravilhoso, li em dois dias', 1),
    ('Bateria dura o dia inteiro, estou satisfeito', 1),
    ('Restaurante bom, preço justo e garçons atenciosos', 1),
    ('Show inesquecível, a banda tocou muito bem', 1),
    ('Curso excelente, professor explica com clareza', 1),
    ('Não me arrependo, produto bonito e resistente', 1),
    ('Serviço rápido e eficiente, parabéns à equipe', 1),
    ('A série é viciante, assisti tudo num fim de semana', 1),
    ('Tênis confortável, ótimo para corrida', 1),
    ('Produto péssimo, quebrou na primeira semana', 0),
    ('Horrível, nunca mais compro nessa loja', 0),
    ('A entrega atrasou e ninguém responde meus e-mails', 0),
    ('Qualidade ruim, não vale o preço', 0),
    ('Fiquei muito decepcionado com o atendimento', 0),
    ('A comida chegou fria e sem sabor', 0),
    ('Filme chato e longo demais, dormi no meio', 0),
    ('Veio com defeito e a troca demorou um mês', 0),
    ('Não gostei, o tecido é fino e desbotou', 0),
    ('O hotel estava sujo e o chuveiro não funcionava', 0),
    ('Aplicativo trava toda hora, uma perda de tempo', 0),
    ('Livro confuso e mal escrito', 0),
    ('A bateria acaba em poucas horas, terrível', 0),
    ('Garçom grosseiro e conta errada', 0),
    ('O som do show estava péssimo', 0),
    ('Curso desorganizado, conteúdo fraco', 0),
    ('Paguei caro por um produto frágil', 0),
    ('Serviço lento e funcionários despreparados', 0),
    ('Série arrastada, o final foi uma decepção', 0),
    ('Tênis apertado e machucou meus pés', 0),
]
textos_teste = [t for t, _ in avaliacoes]
y_teste = np.array([r for _, r in avaliacoes])

# TREINO: 200 avaliações geradas a partir de modelos de frase (rápido de produzir, mas pouco variado)
rng = np.random.default_rng(42)
aspectos = ['Produto', 'Entrega', 'Atendimento', 'Qualidade', 'Preço', 'Comida', 'Filme', 'Hotel',
            'Aplicativo', 'Serviço', 'Livro', 'Curso']
pos_adj = ['excelente', 'ótimo', 'muito bom', 'maravilhoso', 'perfeito', 'incrível', 'nota dez', 'impecável',
           'sensacional', 'rápido e eficiente', 'bom demais', 'confortável e bonito']
neg_adj = ['péssimo', 'horrível', 'muito ruim', 'terrível', 'decepcionante', 'lamentável', 'fraco', 'demorado',
           'uma porcaria', 'abaixo do esperado', 'com defeito', 'caro e ruim']
pos_fecho = ['Recomendo!', 'Adorei.', 'Voltarei a comprar.', 'Estou satisfeito.', 'Gostei muito.',
             'Superou minhas expectativas.', 'Valeu cada centavo.', 'Estou feliz com a compra.']
neg_fecho = ['Não recomendo.', 'Nunca mais compro.', 'Fiquei decepcionado.', 'Quero meu dinheiro de volta.',
             'Não gostei.', 'Que raiva.', 'Perda de tempo.', 'Estou insatisfeito.']
treino = []
for _ in range(100):
    treino.append((f'{rng.choice(aspectos)}: {rng.choice(pos_adj)}. {rng.choice(pos_fecho)}', 1))
    treino.append((f'{rng.choice(aspectos)}: {rng.choice(neg_adj)}. {rng.choice(neg_fecho)}', 0))
textos_treino = [t for t, _ in treino]
y_treino = np.array([r for _, r in treino])
print('Exemplos de treino:', textos_treino[:3])

modelo_sent = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), MultinomialNB(alpha=1.0))
acc_cv = cross_val_score(modelo_sent, textos_treino, y_treino, cv=5).mean()
modelo_sent.fit(textos_treino, y_treino)
pred_nb = modelo_sent.predict(textos_teste)
pred_lex = np.array([1 if analisador.calcular_sentimento(t)['score'] > 0 else 0 for t in textos_teste])

print(f'\nNaive Bayes — validação cruzada nas frases GERADAS : {acc_cv:.0%}')
print(f'Naive Bayes — teste nas 40 avaliações ESCRITAS À MÃO: {(pred_nb == y_teste).mean():.0%}')
print(f'Léxico      — teste nas 40 avaliações ESCRITAS À MÃO: {(pred_lex == y_teste).mean():.0%}')

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, p, nome in [(axs[0], pred_lex, 'Léxico (escrito à mão)'), (axs[1], pred_nb, 'Naive Bayes (aprendido)')]:
    ConfusionMatrixDisplay(confusion_matrix(y_teste, p), display_labels=['neg', 'pos']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nome); ax.grid(False)
plt.tight_layout(); plt.show()

print('Erros do Naive Bayes no teste:')
for t, yv, p in zip(textos_teste, y_teste, pred_nb):
    if yv != p:
        print(f'  real={"pos" if yv else "neg"}  previsto={"pos" if p else "neg"}  ←  "{t}"')

nb = modelo_sent.named_steps['multinomialnb']
termos = modelo_sent.named_steps['tfidfvectorizer'].get_feature_names_out()
razao = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
print('\nTermos mais POSITIVOS aprendidos:', list(termos[np.argsort(razao)[::-1][:8]]))
print('Termos mais NEGATIVOS aprendidos:', list(termos[np.argsort(razao)[:8]]))

#### 🔎 Como interpretar o resultado

**1. A armadilha dos 100%:** na validação cruzada com as frases **geradas**, o Naive Bayes acertou **100%**. Mas essas frases são muito parecidas entre si ("Aspecto: adjetivo. Fecho."), então acertar é fácil. **Avaliar com dados iguais aos de treino superestima o desempenho.**

**2. O teste realista:** nas 40 avaliações **escritas à mão** (vocabulário e estrutura diferentes), o Naive Bayes acertou **85%** e o léxico **82%**. A queda de 100% para 85% é o efeito da **mudança de distribuição** (*distribution shift*): o mundo real não se parece com os dados de treino.

**3. Matrizes de confusão, perfis de erro diferentes:**
- **Léxico:** nunca chamou uma avaliação negativa de positiva, mas errou **7 positivas**. Frases como "Entrega antes do prazo e embalagem caprichada" não têm **nenhuma** palavra do léxico, recebem score 0 e caem em "negativo".
- **Naive Bayes:** 3 erros de cada lado. As frases erradas usam vocabulário **ausente** do treino ("viciante", "caprichada", "fria", "confuso").

**4. O que o modelo realmente aprendeu (últimas linhas):**
- Termos positivos: `cada centavo`, `valeu`, `voltarei comprar`. São os **fechos** das frases geradas.
- Termos negativos: `não`, `nunca mais`, mas também **`de`** e **`que`**! Por quê? Os fechos negativos ("Quero meu dinheiro **de** volta", "**Que** raiva") contêm essas palavras. O modelo aprendeu uma **correlação espúria**, um **artefato** do modo como os dados foram gerados.

- **💡 Lições centrais de ML aplicado a PLN:**
  1. Um modelo aprende **o que está nos dados**, inclusive os vícios deles;
  2. Teste sempre com dados **o mais próximos possível do uso real**;
  3. Mais dados **variados** valem mais que um modelo mais sofisticado. Com milhares de avaliações reais e variadas, o mesmo Naive Bayes melhora muito, e modelos pré-treinados ajustados (como o BERTimbau) vão além, porque já "conhecem" o português antes de ver os exemplos rotulados.

### 4.4.1 Além da acurácia: precisão, revocação e F1

A acurácia esconde **que tipo** de erro o modelo comete. Olhando a classe **positiva**:

- **Precisão** = VP / (VP + FP): *dos que o modelo chamou de positivo, quantos eram mesmo?*
- **Revocação** (*recall*) = VP / (VP + FN): *dos positivos reais, quantos o modelo encontrou?*
- **F1** = média harmônica das duas: 2·P·R / (P + R). Só é alta se **ambas** forem altas.

Um filtro de **spam** precisa de precisão alta (não mandar e-mail importante para o lixo); um detector de **discurso de ódio** ou de **sintomas graves** costuma priorizar revocação (não deixar passar casos reais).

In [ ]:
from sklearn.metrics import classification_report

for nome, pred in [('LÉXICO', pred_lex), ('NAIVE BAYES', pred_nb)]:
    print(f'===== {nome} =====')
    print(classification_report(y_teste, pred, target_names=['negativo', 'positivo'], digits=2, zero_division=0))

#### 🔎 Como interpretar o resultado

- **Léxico:** para a classe *negativo*, a **revocação é 100%** (encontrou todas as negativas) mas a **precisão cai**, porque 7 positivas sem palavras do léxico caíram em "negativo". Para a classe *positivo* é o contrário: **precisão de 100%** (quando diz "positivo", acerta sempre) e **revocação de 65%** (deixou escapar 7 das 20).
- **Naive Bayes:** erros **equilibrados** (3 de cada lado), com precisão e revocação iguais (85%) nas duas classes.
- **Acurácias parecidas (82% × 85%), comportamentos muito diferentes.** Se o objetivo fosse *"encontrar clientes insatisfeitos para a equipe de retenção ligar"*, o léxico nunca deixaria passar uma reclamação, mas faria a equipe ligar para 7 clientes satisfeitos.
- **💡 Conceito-chave:** escolha a métrica pelo **custo do erro** na aplicação. Em PLN, com classes desbalanceadas (spam é minoria, discurso de ódio é minoria), a acurácia sozinha engana: um modelo que diz "não é spam" para tudo tem 95% de acurácia se só 5% dos e-mails forem spam.

### 4.4.2 Naive Bayes "de quadro": a conta completa

O `MultinomialNB` parece uma caixa-preta, mas a conta cabe no quadro. Pelo **Teorema de Bayes** (Aula 06), com a hipótese "ingênua" de que as palavras são independentes dada a classe:

$$P(c \mid \text{doc}) \;\propto\; P(c) \times \prod_{w \in \text{doc}} P(w \mid c), \qquad P(w \mid c) = \frac{\text{contagem}(w, c) + 1}{\text{total de palavras em } c + |V|}$$

O **+1** é a **suavização de Laplace**: sem ela, uma palavra que nunca apareceu em uma classe zeraria o produto inteiro. Treino com 6 frases:

| Positivas | Negativas |
|---|---|
| adorei o filme | filme chato |
| filme ótimo adorei | odiei o filme chato |
| ótimo elenco | péssimo elenco |

In [ ]:
# Naive Bayes multinomial implementado à mão, com todas as contas visíveis
treino_nb = [('adorei o filme', 'pos'), ('filme ótimo adorei', 'pos'), ('ótimo elenco', 'pos'),
             ('filme chato', 'neg'), ('odiei o filme chato', 'neg'), ('péssimo elenco', 'neg')]
classes = ['pos', 'neg']
V_nb = sorted({w for t, _ in treino_nb for w in t.split()})
contagem = {c: Counter(w for t, cl in treino_nb if cl == c for w in t.split()) for c in classes}
total = {c: sum(contagem[c].values()) for c in classes}
prior = {c: sum(cl == c for _, cl in treino_nb) / len(treino_nb) for c in classes}

def p_palavra(w, c):
    return (contagem[c][w] + 1) / (total[c] + len(V_nb))      # suavização de Laplace

print(f'|V| = {len(V_nb)} palavras | palavras em pos = {total["pos"]}, em neg = {total["neg"]} | P(pos) = P(neg) = 0,5\n')
print(f'{"palavra":<9}{"cont pos":>9}{"cont neg":>9}{"P(w|pos)":>10}{"P(w|neg)":>10}')
for w in V_nb:
    print(f'{w:<9}{contagem["pos"][w]:>9}{contagem["neg"][w]:>9}{p_palavra(w, "pos"):>10.3f}{p_palavra(w, "neg"):>10.3f}')

def classificar_nb(texto):
    palavras = [w for w in texto.split() if w in V_nb]          # palavras desconhecidas são ignoradas
    score = {c: prior[c] * np.prod([p_palavra(w, c) for w in palavras]) for c in classes}
    z = sum(score.values())
    return palavras, score, {c: score[c] / z for c in classes}

for frase in ['adorei o elenco', 'filme chato demais', 'o filme']:
    usadas, score, post = classificar_nb(frase)
    print(f'\n"{frase}"  (palavras usadas: {usadas})')
    for c in classes:
        contas = ' × '.join(f'{p_palavra(w, c):.3f}' for w in usadas)
        print(f'   P({c})·Π = 0,5 × {contas} = {score[c]:.5f}')
    print(f'   → normalizando: P(pos|doc) = {post["pos"]:.1%}   P(neg|doc) = {post["neg"]:.1%}')

# Conferindo com o scikit-learn (CountVectorizer + MultinomialNB com alpha=1 = Laplace)
cv_nb = CountVectorizer(token_pattern=r'\S+').fit([t for t, _ in treino_nb])
sk_nb = MultinomialNB(alpha=1.0).fit(cv_nb.transform([t for t, _ in treino_nb]), [c for _, c in treino_nb])
print('\nscikit-learn, P(pos|"adorei o elenco") =', f'{sk_nb.predict_proba(cv_nb.transform(["adorei o elenco"]))[0][list(sk_nb.classes_).index("pos")]:.1%}')

#### 🔎 Como interpretar o resultado

- **A tabela de probabilidades é o "modelo" inteiro.** Treinar um Naive Bayes é só **contar palavras por classe**. Por isso ele é tão rápido e foi, por anos, o padrão em filtros de spam (Paul Graham, *A Plan for Spam*, 2002).
- **Suavização em ação:** `odiei` nunca aparece em frases positivas, mas P(odiei|pos) = 1/(8+8) = 0,062, e não zero. Sem o +1, qualquer frase positiva com a palavra *odiei* teria probabilidade **zero** de ser positiva, não importa o resto.
- **"adorei o elenco" → 75% positivo:** `adorei` pesa a favor (0,188 contra 0,062, ou seja, **3×** mais provável em textos positivos); `o` e `elenco` aparecem uma vez em cada classe, têm a mesma probabilidade nas duas e **se cancelam** na normalização. A razão 3:1 vira 75% × 25%.
- **"filme chato demais" → 75% negativo:** `demais` **não está no vocabulário** e é simplesmente ignorada. `chato` decide (razão 3:1 a favor de negativo). `filme` aparece duas vezes em cada classe e não pesa para nenhum lado.
- **"o filme" → 50/50:** as duas palavras são **neutras** neste treino. O modelo não tem evidência e volta à probabilidade *a priori*.
- **O scikit-learn dá exatamente o mesmo número**, confirmando que a biblioteca faz a mesma conta.
- **⚠️ Na prática, usamos logaritmos:** multiplicar centenas de probabilidades pequenas causa *underflow* (o resultado vira 0.0 no computador). Somamos log P(w|c) em vez de multiplicar. É o que o `MultinomialNB` faz por dentro (`feature_log_prob_`, usado na Seção 4.4).
- **💡 Por que "ingênuo"?** A hipótese de independência é claramente falsa ("não" e "gostei" não são independentes), mas o classificador funciona bem porque, para **decidir a classe**, basta que a ordem das probabilidades esteja certa, não os valores exatos.

### 4.5 Modelos de linguagem: prevendo a próxima palavra

Um **modelo de linguagem** atribui probabilidades a sequências de palavras. O mais simples é o de **bigramas**, que usa só a palavra anterior:

$$P(w_1, \dots, w_n) \approx \prod_{i} P(w_i \mid w_{i-1}), \qquad P(w_i \mid w_{i-1}) = \frac{\text{contagem}(w_{i-1}, w_i)}{\text{contagem}(w_{i-1})}$$

Os grandes modelos de linguagem (GPT, Claude, LLaMA) fazem **a mesma tarefa**, prever a próxima palavra (*token*), mas olhando para **milhares** de palavras anteriores com redes neurais *Transformer*.

In [ ]:
# Modelo de linguagem de BIGRAMAS: P(próxima palavra | palavra atual), estimado por contagem no corpus
tokens_lm = re.findall(r'[a-záéíóúâêôãõàçü]+|[.!?]', corpus_machado.lower())
bigramas = Counter(zip(tokens_lm, tokens_lm[1:]))
seguintes = {}
for (a, b), n in bigramas.items():
    seguintes.setdefault(a, Counter())[b] = n

def prob_proxima(palavra, k=8):
    c = seguintes.get(palavra, Counter())
    total = sum(c.values())
    return [(w, n / total) for w, n in c.most_common(k)] if total else []

for w in ['minha', 'olhos', 'capitu']:
    print(f'P( · | "{w}") →', ', '.join(f'{p} {pr:.1%}' for p, pr in prob_proxima(w, 6)))

def gerar(inicio, n=18, semente=7, temperatura=1.0):
    rng = np.random.default_rng(semente)
    texto, historico = [inicio], []
    for _ in range(n):
        c = seguintes.get(texto[-1])
        if not c:
            break
        palavras, contagens = zip(*c.most_common(30))
        p = np.array(contagens, dtype=float) ** (1 / temperatura); p /= p.sum()
        escolha = palavras[rng.choice(len(palavras), p=p)]
        historico.append((texto[-1], palavras[:8], p[:8], escolha))
        texto.append(escolha)
    return ' '.join(texto), historico

inicio_lm = 'capitu' if 'capitu' in seguintes else tokens_lm[0]
frase_gerada, hist_lm = gerar(inicio_lm)
print('\nTexto gerado (bigramas, amostragem):', frase_gerada)

#### 🔎 Como interpretar o resultado

- **`P(· | "minha")`:** depois de "minha", a palavra mais provável em Machado é **"mãe"** (19%), seguida de "vida", "filha", "parte", "alma"... As probabilidades refletem o **estilo** e os **temas** do autor: família, alma, vida interior.
- **`P(· | "olhos")` e `P(· | "capitu")`:** o mais provável é o **ponto final** (`.`). "Olhos" e "Capitu" costumam **terminar** frases. Lembre-se dos "olhos de ressaca" de Capitu: "olhos de" tem 8,7%.
- **Texto gerado:** *"capitu ia até às vezes e uma vez os lábios e o seu próprio que se ao tempo para"*.
  - **Localmente** faz sentido: cada par de palavras vizinhas é plausível ("às vezes", "uma vez", "os lábios", "seu próprio");
  - **Globalmente** não faz sentido: o modelo só "lembra" **uma** palavra para trás, então perde o fio da frase.
- **Por que sortear em vez de pegar sempre a mais provável?** Escolher sempre o máximo (*greedy*) gera repetições ("de que de que de que..."). A **amostragem** produz variedade. O parâmetro `temperatura` controla isso: temperatura baixa = mais conservador, alta = mais criativo (e mais incoerente). LLMs como GPT e Claude usam exatamente esse mecanismo.
- **💡 Conceito-chave:** um bigrama é um LLM com **contexto de 1 palavra** e "parâmetros" que são contagens. Os modelos atuais têm contexto de **centenas de milhares** de tokens e bilhões de parâmetros aprendidos, mas a tarefa de treino é a mesma: **prever o próximo token**.

In [ ]:
# 🎬 ANIMAÇÃO — o modelo escolhendo cada próxima palavra
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.5), gridspec_kw={'height_ratios': [1, 2.2]})

def quadro_lm(k):
    ax1.clear(); ax2.clear(); ax1.axis('off')
    atual, cands, probs, escolha = hist_lm[k]
    ja = ' '.join([hist_lm[0][0]] + [h[3] for h in hist_lm[:k]])
    ax1.text(0, 0.5, ja + ' ', fontsize=13, va='center', wrap=True)
    ax1.set_title('Texto gerado até agora (a última palavra é o contexto)', fontsize=10)
    cores = [CORES['laranja'] if c == escolha else CORES['azul'] for c in cands]
    ax2.bar(cands, probs, color=cores)
    ax2.set_ylabel('probabilidade'); ax2.set_ylim(0, max(0.5, max(probs) * 1.1))
    ax2.set_title(f'P(próxima | "{atual}") — top 8. Sorteada: "{escolha}" (laranja)', fontsize=11)
    ax2.tick_params(axis='x', rotation=20)
    fig.tight_layout()

anim = animation.FuncAnimation(fig, quadro_lm, frames=len(hist_lm), interval=1200)
mostrar_animacao(anim)

#### 🔎 Como interpretar a animação

- **Painel de cima:** o texto gerado até agora. A **última palavra** é o único contexto que o modelo enxerga.
- **Painel de baixo:** as 8 próximas palavras mais prováveis dado esse contexto. A barra **laranja** é a palavra sorteada.
- **O que observar:**
  - Depois de palavras funcionais ("e", "o", "de"), a distribuição é **achatada**: muitas continuações possíveis e grande incerteza;
  - Depois de palavras como "às", a distribuição é mais **concentrada** ("vezes" tem 19%, quatro vezes mais que a segunda opção);
  - Às vezes a palavra sorteada **não** é a mais provável. É o efeito da amostragem, e é por isso que o texto varia a cada `semente`.
- **💡 Experimente:** na célula anterior, mude `gerar(inicio_lm)` para `gerar('minha', temperatura=0.5)` e depois para `temperatura=2.0`. Compare a coerência dos textos.

### 4.5.1 Avaliando um modelo de linguagem: perplexidade e suavização

Como saber se um modelo de linguagem é bom? Damos a ele um texto que ele **nunca viu** e medimos o quanto ele se "surpreende". A métrica padrão é a **perplexidade**:

$$PP = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \ln P(w_i \mid w_{i-1})\right)$$

Intuição: **perplexidade = número médio de palavras entre as quais o modelo está "em dúvida"** a cada passo. Um modelo que escolhe ao acaso entre 10 mil palavras tem PP = 10.000; um modelo perfeito tem PP = 1. **Quanto menor, melhor.**

Problema: se um bigrama do teste **nunca apareceu** no treino, P = 0, ln 0 = −∞ e a perplexidade explode. A solução é a **suavização**: reservar um pouco de probabilidade para o que não foi visto.

- **Add-k (Laplace generalizado):** $P(b \mid a) = \dfrac{c(a, b) + k}{c(a) + k\,|V|}$
- **Interpolação:** $P(b \mid a) = \lambda \, P_{\text{bigrama}}(b \mid a) + (1-\lambda)\, P_{\text{unigrama}}(b)$, misturando um modelo com mais contexto e outro mais robusto.

In [ ]:
# Perplexidade em texto NÃO VISTO: treinamos em 90% de Machado de Assis e testamos nos 10% finais
corte = int(len(tokens_lm) * 0.9)
treino_lm, teste_lm = tokens_lm[:corte], tokens_lm[corte:]
vocab_lm = {w for w, n in Counter(treino_lm).items() if n >= 2}          # palavras raras → <unk>
unk = lambda seq: [w if w in vocab_lm else '<unk>' for w in seq]
treino_lm, teste_lm = unk(treino_lm), unk(teste_lm)
V_lm = len(vocab_lm) + 1
uni_lm = Counter(treino_lm)
bi_lm = Counter(zip(treino_lm, treino_lm[1:]))
N_treino = len(treino_lm)
pares_teste = list(zip(teste_lm, teste_lm[1:]))

def perplexidade(p_func):
    logs = [math.log(p_func(a, b)) for a, b in pares_teste]
    return math.exp(-sum(logs) / len(logs))

p_unigrama = lambda a, b: (uni_lm[b] + 1) / (N_treino + V_lm)
p_add_k = lambda k: (lambda a, b: (bi_lm[(a, b)] + k) / (uni_lm[a] + k * V_lm))
p_interp = lambda lam: (lambda a, b: lam * (bi_lm[(a, b)] / uni_lm[a] if uni_lm[a] else 0) + (1 - lam) * p_unigrama(a, b))

nao_vistos = sum(bi_lm[par] == 0 for par in pares_teste) / len(pares_teste)
print(f'Treino: {len(treino_lm):,} tokens | teste: {len(teste_lm):,} tokens | |V| = {V_lm:,}')
print(f'Bigramas do teste que NUNCA apareceram no treino: {nao_vistos:.1%}  → sem suavização, P = 0 e PP = ∞\n')

resultados_pp = {'Uniforme (chute)': V_lm, 'Unigrama (sem contexto)': perplexidade(p_unigrama)}
for k in [1, 0.1, 0.01]:
    resultados_pp[f'Bigrama add-k (k={k})'] = perplexidade(p_add_k(k))
for lam in [0.5, 0.7, 0.9]:
    resultados_pp[f'Bigrama interpolado (λ={lam})'] = perplexidade(p_interp(lam))
for nome, pp in resultados_pp.items():
    print(f'  {nome:<30} perplexidade = {pp:8,.0f}')

fig, ax = plt.subplots(figsize=(9, 3.6))
nomes = list(resultados_pp)[1:]
ax.barh(nomes, [resultados_pp[n] for n in nomes], color=[CORES['cinza']] + [CORES['laranja']] * 3 + [CORES['verde']] * 3)
ax.invert_yaxis(); ax.set_xlabel('perplexidade no teste (menor = melhor)')
ax.set_title('Mais contexto ajuda, mas só com uma boa suavização'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

#### 🔎 Como interpretar o resultado

- **1 em cada 4 bigramas do teste (~25%) nunca apareceu no treino**, mesmo com mais de meio milhão de palavras de Machado de Assis. Sem suavização, o modelo daria probabilidade **zero** a qualquer frase nova com um desses pares. É a Lei de Zipf de novo: a maioria das combinações possíveis é **rara**.
- **Uniforme (15 mil):** chutar entre todas as palavras do vocabulário. É o pior caso possível.
- **Unigrama (~600):** só saber **quais palavras são frequentes** já reduz a dúvida 25 vezes.
- **Bigrama add-1 (~1.800), pior que o unigrama!** Somar 1 a **todos** os 15 mil × 15 mil pares possíveis tira probabilidade demais dos pares vistos e a espalha pelos não vistos. Diminuir *k* melhora (k = 0,01 → ~550), mas a suavização *add-k* é sempre grosseira.
- **Interpolação (~310, a melhor):** misturar o bigrama (preciso quando o par é conhecido) com o unigrama (robusto quando não é) **corta a perplexidade pela metade** em relação ao unigrama. Com λ = 0,9 o modelo confia demais no bigrama e piora.
- **💡 Conceitos-chave:**
  1. Avalie modelos de linguagem em **texto não visto** (aqui, os capítulos finais), como qualquer modelo de ML;
  2. **Nunca dê probabilidade zero**: a língua é criativa e sempre haverá combinações novas;
  3. **Mais contexto só ajuda se houver dados** para estimá-lo. Trigramas teriam ainda mais pares não vistos. Essa é a limitação que levou às **redes neurais**: em vez de contar pares exatos, elas **generalizam** entre palavras parecidas (*embeddings*), e por isso os LLMs atingem perplexidades muito menores.

### 4.6 Semântica distribucional: "diga-me com quem andas..."

A **hipótese distribucional** (Harris, 1954; Firth, 1957: *"You shall know a word by the company it keeps"*) afirma que palavras que aparecem em **contextos parecidos** têm **significados parecidos**. É a ideia por trás do Word2Vec (2013) e dos *embeddings* dos LLMs.

Vamos construir *embeddings* "à mão": contar quais palavras aparecem perto de quais (matriz de **coocorrência**), ponderar com **PPMI** (*Positive Pointwise Mutual Information*) e comprimir com **SVD**.

In [ ]:
# Semântica distribucional: representar cada palavra pelos CONTEXTOS em que ela aparece
rng = np.random.default_rng(0)
categorias = {
    'animal':  ['gato', 'cachorro', 'cavalo', 'coelho', 'pato'],
    'comida':  ['pão', 'arroz', 'bolo', 'queijo', 'feijão'],
    'cidade':  ['recife', 'curitiba', 'manaus', 'salvador', 'natal'],
    'veículo': ['carro', 'ônibus', 'caminhão', 'metrô', 'trem'],
}
modelos_frase = {
    'animal':  ['o {} dorme no quintal', 'meu {} late e corre', 'o {} come ração', 'levei o {} ao veterinário', 'o {} brinca no jardim'],
    'comida':  ['comi {} no almoço', 'o {} está quente e gostoso', 'comprei {} na padaria', 'o {} saiu do forno', 'preparei {} para o jantar'],
    'cidade':  ['viajei para {} nas férias', 'o aeroporto de {} lotou', 'moro em {} desde criança', 'o prefeito de {} falou', 'as praias de {} são lindas'],
    'veículo': ['o {} quebrou na estrada', 'fui de {} ao trabalho', 'o {} estava lotado', 'troquei o pneu do {}', 'estacionei o {} na garagem'],
}
frases_emb = [rng.choice(modelos_frase[c]).format(p) for _ in range(30) for c, ps in categorias.items() for p in ps]
print(f'{len(frases_emb)} frases geradas. Exemplos:', frases_emb[:4])

# Matriz de coocorrência (janela de ±2 palavras) → PPMI → SVD (2 dimensões)
toks_emb = [f.split() for f in frases_emb]
vocab_emb = sorted({w for f in toks_emb for w in f})
idx = {w: i for i, w in enumerate(vocab_emb)}
Co = np.zeros((len(vocab_emb), len(vocab_emb)))
for f in toks_emb:
    for i, w in enumerate(f):
        for j in range(max(0, i - 2), min(len(f), i + 3)):
            if i != j:
                Co[idx[w], idx[f[j]]] += 1
P = Co / Co.sum()
pmi = np.log(np.maximum(P / (P.sum(1, keepdims=True) * P.sum(0, keepdims=True)), 1e-12))
PPMI = np.maximum(pmi, 0)
U, S, Vt = np.linalg.svd(PPMI)
vetores = U[:, :20] * S[:20]                          # "embedding" de 20 dimensões por palavra

def vizinhos(w, k=4):
    v = vetores[idx[w]]
    sims = vetores @ v / (np.linalg.norm(vetores, axis=1) * np.linalg.norm(v) + 1e-12)
    return [(vocab_emb[i], sims[i]) for i in np.argsort(-sims) if vocab_emb[i] != w][:k]

for w in ['gato', 'bolo', 'recife', 'trem']:
    print(f'mais parecidas com "{w}":', ', '.join(f'{p} ({s:.2f})' for p, s in vizinhos(w)))

pal = [p for ps in categorias.values() for p in ps]
cor_cat = {'animal': CORES['laranja'], 'comida': CORES['verde'], 'cidade': CORES['azul'], 'veículo': CORES['roxo']}
# Para desenhar: normalizamos os vetores das 20 palavras-alvo e projetamos em 2D com PCA
alvos = np.array([vetores[idx[p]] / np.linalg.norm(vetores[idx[p]]) for p in pal])
alvos_c = alvos - alvos.mean(0)
_, _, Vp = np.linalg.svd(alvos_c, full_matrices=False)
proj = dict(zip(pal, alvos_c @ Vp[:2].T))
fig, ax = plt.subplots(figsize=(8, 6))
for c, ps in categorias.items():
    pts = np.array([proj[p] for p in ps])
    ax.scatter(pts[:, 0], pts[:, 1], s=80, color=cor_cat[c], label=c)
    for k, (p, (x, y)) in enumerate(zip(ps, pts)):     # rótulos espaçados: os pontos quase coincidem
        ax.annotate(p, (x, y), xytext=(14, 13 * (k - 2)), textcoords='offset points', fontsize=10,
                    arrowprops=dict(arrowstyle='-', color='#999', lw=0.6))
ax.legend(); ax.set_title('Embeddings (SVD da matriz PPMI) das 20 palavras, projetados em 2D')
ax.set_xlabel('dimensão 1'); ax.set_ylabel('dimensão 2'); ax.grid(alpha=0.3); ax.margins(0.25)
plt.show()

#### 🔎 Como interpretar o resultado

- **As frases** foram geradas a partir de modelos por categoria. Ninguém disse ao algoritmo que "gato" é um animal: ele só vê **quais palavras aparecem perto de quais**.
- **Vizinhos mais próximos** (similaridade de cosseno entre os vetores):
  - `gato` → pato, cachorro, cavalo, coelho (**todos animais**);
  - `bolo` → pão, queijo, arroz, feijão (**todos comidas**);
  - `recife` → salvador, natal, manaus, curitiba (**todas cidades**);
  - `trem` → ônibus, caminhão, metrô, carro (**todos veículos**).
- **Gráfico:** as 20 palavras formam **4 grupos** bem separados, um por categoria. Dentro de cada grupo os pontos quase coincidem, porque neste corpus artificial as palavras de uma categoria aparecem em contextos **idênticos** (os rótulos foram espaçados para ficarem legíveis).
- **Por que funciona?**
  1. A **matriz de coocorrência** conta, para cada palavra, quais palavras aparecem a até 2 posições dela;
  2. A **PPMI** destaca coocorrências **mais frequentes do que o acaso** e descarta as triviais (como "o" com tudo);
  3. A **SVD** comprime essa matriz em poucas dimensões, juntando contextos que se repetem juntos. Palavras com contextos parecidos ficam com vetores parecidos.
- **💡 Conceito-chave:** é a **hipótese distribucional** na prática: *significado = padrão de uso*. O Word2Vec (2013) aprende vetores com o mesmo princípio usando uma rede neural, e os Transformers vão além, com vetores que mudam conforme o **contexto da frase** ("banco" da praça × "banco" financeiro).
- **⚠️ Limitação:** num corpus real e pequeno, os grupos seriam bem mais "bagunçados". *Embeddings* de qualidade exigem **milhões** de frases.

### 4.7 Do Word2Vec ao Transformer: *embeddings* contextuais e atenção

Os *embeddings* da Seção 4.6 (e do Word2Vec) são **estáticos**: cada palavra tem **um único vetor**, seja qual for a frase. "Banco" recebe o mesmo vetor em *"sentei no banco da praça"* e em *"depositei dinheiro no banco"*. É uma média dos dois sentidos.

O **Transformer** (Vaswani et al., 2017) resolve isso com a **autoatenção** (*self-attention*): o vetor de cada palavra é **recalculado** como uma média ponderada dos vetores de **todas** as palavras da frase, com pesos maiores para as mais **relevantes**:

$$\text{Atenção}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V$$

- **Q (*query*, consulta):** "o que esta palavra está procurando?"
- **K (*key*, chave):** "o que esta palavra oferece?"
- **V (*value*, valor):** "a informação que esta palavra passa adiante"
- **softmax(Q·Kᵀ/√d):** a **semelhança** entre consulta e chave vira um peso entre 0 e 1 (cada linha soma 1). O √d evita que os produtos fiquem grandes demais.

Em um Transformer real, Q, K e V são obtidos multiplicando os *embeddings* por matrizes **aprendidas** ($W_Q, W_K, W_V$), há várias "cabeças" de atenção em paralelo e dezenas de camadas empilhadas. No exemplo abaixo usamos a versão mais simples possível ($Q = K = V$ = os próprios vetores) com vetores de 4 dimensões escritos à mão, para enxergar o mecanismo.

In [ ]:
# Autoatenção "de brinquedo": como o vetor de "banco" muda conforme a frase
# Dimensões (interpretáveis só neste exemplo): [lugar/natureza, dinheiro, assento/móvel, palavra funcional]
E = {'sentei':    [0.2, 0.0, 1.0, 0.0], 'praça':  [1.0, 0.0, 0.3, 0.0],
     'depositei': [0.0, 1.0, 0.0, 0.0], 'dinheiro': [0.0, 1.0, 0.0, 0.0],
     'banco':     [0.5, 0.6, 0.5, 0.0], 'no': [0.0, 0.0, 0.0, 1.0], 'da': [0.0, 0.0, 0.0, 1.0]}
E = {w: np.array(v) for w, v in E.items()}

def autoatencao(frase):
    palavras = frase.split()
    X = np.array([E[w] for w in palavras])
    d = X.shape[1]
    escores = X @ X.T / np.sqrt(d)                                  # Q·Kᵀ/√d  (aqui Q = K = V = X)
    pesos = np.exp(escores) / np.exp(escores).sum(1, keepdims=True)  # softmax por linha
    return palavras, pesos, pesos @ X                                # nova representação = média ponderada

cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
frases_att = ['sentei no banco da praça', 'depositei dinheiro no banco']
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, frase in zip(axs, frases_att):
    palavras, pesos, Z = autoatencao(frase)
    ax.imshow(pesos, cmap='Oranges', vmin=pesos.min() - 0.03, vmax=pesos.max())
    ax.set_xticks(range(len(palavras)), palavras, rotation=30); ax.set_yticks(range(len(palavras)), palavras)
    for i in range(len(palavras)):
        for j in range(len(palavras)):
            ax.text(j, i, f'{pesos[i, j]:.2f}', ha='center', va='center', fontsize=9)
    ax.set_title(f'"{frase}"\n(linha = quem olha, coluna = para quem olha)'); ax.grid(False)
    b = palavras.index('banco')
    print(f'"{frase}"')
    print('   atenção de "banco" →', ', '.join(f'{p} {w:.2f}' for p, w in zip(palavras, pesos[b])))
    print(f'   "banco" ANTES  : sim(praça) = {cos(E["banco"], E["praça"]):.2f} | sim(dinheiro) = {cos(E["banco"], E["dinheiro"]):.2f}')
    print(f'   "banco" DEPOIS : sim(praça) = {cos(Z[b], E["praça"]):.2f} | sim(dinheiro) = {cos(Z[b], E["dinheiro"]):.2f}\n')
plt.tight_layout(); plt.show()

#### 🔎 Como interpretar o resultado

- **Antes da atenção**, "banco" é igualmente parecido com *praça* e com *dinheiro* (vetor estático, ambíguo).
- **Depois da atenção**, o vetor de "banco" **se desloca na direção do contexto**:
  - em *"sentei no banco da praça"*, fica **mais parecido com "praça"**, porque recebeu parte dos vetores de *praça* e *sentei*;
  - em *"depositei dinheiro no banco"*, fica **mais parecido com "dinheiro"**, porque recebeu parte de *dinheiro* e *depositei*.
- **Mapas de calor:** cada linha mostra para onde uma palavra "olha" (os pesos somam 1). As palavras funcionais (`no`, `da`) olham sobretudo uma para a outra: são parecidas entre si e diferentes das palavras de conteúdo. Em um modelo treinado, cabeças diferentes aprendem padrões diferentes (sujeito → verbo, pronome → referente, adjetivo → substantivo...).
- **O que falta para ser um Transformer de verdade:**
  1. matrizes **aprendidas** $W_Q, W_K, W_V$ (o modelo aprende *o que* procurar, em vez de usar a similaridade crua);
  2. **codificação de posição** (sem ela, "o gato persegue o rato" e "o rato persegue o gato" seriam iguais, o mesmo problema do BoW!);
  3. **várias cabeças** e **dezenas de camadas**, intercaladas com redes *feed-forward*;
  4. **treino** em bilhões de tokens, prevendo a próxima palavra (GPT) ou palavras mascaradas (BERT).
- **💡 Conceito-chave:** atenção = **"diga-me com quem andas"** aplicado **dentro de cada frase**. A hipótese distribucional (Seção 4.6) dá o significado médio de uma palavra; a atenção ajusta esse significado ao contexto **daquela ocorrência**. Isso resolve boa parte da **ambiguidade lexical** da Seção 1.5.

### 4.8 Juntando as peças: como funciona um LLM

Quase tudo que vimos nesta aula reaparece, em escala gigante, nos grandes modelos de linguagem (*Large Language Models*):

| Conceito desta aula | Onde aparece em um LLM (GPT, Claude, LLaMA, Gemini) |
|---|---|
| **Tokenização por subpalavras** (BPE, 2.8) | O texto vira uma sequência de *tokens* com vocabulário de ~50 a 200 mil subpalavras. É por isso que LLMs às vezes erram ao contar letras: eles não "veem" letras, veem tokens |
| **Lei de Zipf** (2.7) | Poucos tokens são muito frequentes e há uma cauda longa de raros. Dados raros (línguas, domínios) são aprendidos pior |
| ***Embeddings*** (4.6) | Cada token vira um vetor de milhares de dimensões, aprendido no treino |
| **Atenção** (4.7) | Dezenas de camadas de autoatenção ajustam cada vetor ao contexto, com janelas de centenas de milhares de tokens |
| **Modelo de linguagem** (4.5) | A tarefa de **pré-treino** é a mesma do bigrama: prever o próximo token, só que olhando o contexto inteiro |
| **Perplexidade** (4.5.1) | É a métrica acompanhada durante o pré-treino |
| **Temperatura e amostragem** (4.5) | Controlam a geração: temperatura baixa = respostas mais previsíveis; alta = mais variadas |
| **Busca com TF-IDF / cosseno** (3.5) | Em **RAG** (*Retrieval-Augmented Generation*), um buscador encontra documentos relevantes e os coloca no *prompt* para o LLM responder com base neles |
| **Aprender com exemplos rotulados** (4.4) | ***Fine-tuning*** com instruções e **RLHF** (aprendizado por reforço com feedback humano) transformam um "completador de texto" em um assistente |

**As três fases de treino de um assistente moderno:**

1. **Pré-treino:** prever o próximo token em trilhões de tokens de texto. O modelo aprende gramática, fatos, estilo e raciocínio **implícito** nos dados. Resultado: um "completador de texto" muito bom.
2. **Ajuste por instruções (*instruction tuning*):** treino supervisionado com pares *(pergunta, boa resposta)* escritos por pessoas. O modelo aprende a **seguir instruções**.
3. **Alinhamento (RLHF e variantes):** pessoas (ou outros modelos) comparam respostas; o modelo é ajustado para preferir respostas úteis, corretas e seguras.

**Limitações importantes (e ligadas ao que vimos):**

- **Alucinação:** o modelo gera o texto **mais plausível**, não necessariamente o **verdadeiro**. Lembre do bigrama: frases localmente plausíveis e globalmente sem sentido. Os LLMs são muito melhores, mas o princípio é o mesmo.
- **Viés:** como o Naive Bayes da Seção 4.4 aprendeu correlações espúrias ("de", "que" = negativo), um LLM aprende os **vieses** dos textos de treino.
- **Conhecimento congelado:** o modelo só "sabe" o que estava nos dados até a data de corte, daí o uso de RAG e de ferramentas de busca.

## 5. Bibliotecas de PLN

### 5.1 NLTK (Natural Language Toolkit)

**NLTK** é biblioteca Python clássica para PLN, com ferramentas para:
- Tokenização
- Stemming e lemmatização
- POS tagging (marcação de classes gramaticais)
- Parsing sintático
- Corpora e léxicos

```python
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer  # Stemmer para português

# Download de recursos
# nltk.download('punkt')
# nltk.download('rslp')

texto = "Os gatos estão correndo rapidamente"
tokens = word_tokenize(texto, language='portuguese')
print(tokens)

stemmer = RSLPStemmer()
stems = [stemmer.stem(token) for token in tokens]
print(stems)
```

### 5.2 spaCy

**spaCy** é biblioteca moderna e eficiente, focada em produção:
- Pipeline de processamento rápido
- Modelos pré-treinados para múltiplos idiomas
- NER (Named Entity Recognition)
- Análise de dependências

```python
import spacy

# Carrega modelo para português
# python -m spacy download pt_core_news_sm
nlp = spacy.load('pt_core_news_sm')

texto = "Apple está construindo fábrica em São Paulo"
doc = nlp(texto)

# Entidades nomeadas
for ent in doc.ents:
    print(ent.text, ent.label_)

# Análise morfológica
for token in doc:
    print(token.text, token.pos_, token.lemma_)
```

### 5.3 Transformers (Hugging Face)

**Transformers** oferece acesso a modelos estado-da-arte:
- BERT, GPT, RoBERTa, etc.
- Fine-tuning para tarefas específicas
- Modelos multilíngues

```python
from transformers import pipeline

# Pipeline de análise de sentimentos
classificador = pipeline('sentiment-analysis', 
                        model='neuralmind/bert-base-portuguese-cased')

resultado = classificador("Este produto é excelente!")
print(resultado)
```

> ⚠️ `neuralmind/bert-base-portuguese-cased` (BERTimbau) é um modelo **base**, pré-treinado apenas para entender a língua. Para análise de sentimentos ele precisa antes ser **ajustado** (*fine-tuning*) com textos rotulados, ou você deve escolher no Hugging Face Hub um modelo já ajustado para sentimentos em português.

## 6. Aplicações de PLN

### 6.1 Chatbots e Assistentes Virtuais
- Siri, Alexa, Google Assistant
- Atendimento ao cliente automatizado
- FAQ bots

### 6.2 Tradução Automática
- Google Translate, DeepL
- Modelos neurais (Transformer-based)
- Tradução em tempo real

### 6.3 Análise de Sentimentos
- Monitoramento de redes sociais
- Análise de reviews de produtos
- Pesquisas de opinião

### 6.4 Extração de Informação
- Named Entity Recognition (NER)
- Extração de relações
- Knowledge graphs

### 6.5 Sumarização de Texto
- Resumos automáticos de notícias
- TL;DR (Too Long; Didn't Read)
- Extração de pontos-chave

### 6.6 Sistemas de Pergunta-Resposta
- IBM Watson
- Busca semântica
- Assistentes de pesquisa

### 6.7 Classificação de Documentos
- Filtragem de spam
- Categorização de notícias
- Triagem de documentos legais

### 6.8 Geração de Texto
- GPT-3, GPT-4 (geração criativa)
- Completar código (GitHub Copilot)
- Geração de relatórios

### 6.9 Reconhecimento de Fala
- Speech-to-text
- Transcrição automática
- Comandos de voz

### 6.10 Análise Sintática e Semântica
- Correção gramatical
- Análise de estilo
- Detecção de plágio

## 7. Comparação de Abordagens em PLN

| Critério | Baseada em Regras | Estatística/ML | Deep Learning |
|----------|-------------------|----------------|---------------|
| **Conhecimento linguístico** | Essencial | Opcional | Mínimo |
| **Dados necessários** | Poucos | Moderados | Muitos |
| **Interpretabilidade** | Alta | Moderada | Baixa |
| **Manutenção** | Difícil (regras manuais) | Moderada | Fácil (retreino) |
| **Generalização** | Baixa | Boa | Excelente |
| **Custo computacional** | Baixo | Moderado | Alto |
| **Precisão** | Variável | Boa | Excelente |
| **Domínios específicos** | Excelente | Boa | Requer dados |
| **Multilíngue** | Difícil | Possível | Facilitado |
| **Exemplos** | Expressões regulares | Naive Bayes, SVM | BERT, GPT, Transformers |
| **Época predominante** | 1960s-1980s | 1990s-2010s | 2010s-presente |

## ✅ Resumo da aula: o que levar para casa

| Problema | Técnica | Ideia em uma frase | Seção |
|---|---|---|:-:|
| Linguagem é ambígua | — | O mesmo texto tem várias leituras; contexto e conhecimento de mundo desfazem a ambiguidade | 1.4–1.5 |
| Conversar com regras | ELIZA / regex | Casamento de padrões imita conversa sem entender nada; um LLM gera a resposta em vez de escolher uma pronta | 1.6–1.8, 2.6 |
| Separar e normalizar palavras | Tokenização, *stopwords*, *stemming*, lematização | Reduz o vocabulário juntando variantes, sempre perdendo alguma informação | 2.1–2.5 |
| Entender a estatística da língua | Lei de Zipf | Poucas palavras são muito frequentes; a maioria é rara | 2.7 |
| Palavras desconhecidas | BPE | Quebrar palavras em subpalavras frequentes | 2.8 |
| Erros de digitação | Distância de Levenshtein | Programação dinâmica conta inserções, remoções e substituições | 2.9 |
| Texto → números | BoW, n-gramas, TF-IDF | Contar palavras e dar mais peso às raras e discriminantes | 3 |
| Buscar documentos | Modelo vetorial + cosseno | Documento e consulta viram vetores; ordena-se pelo ângulo | 3.5 |
| Classificar opinião | Léxico × Naive Bayes | Regras escritas à mão × probabilidades contadas nos dados | 4.1–4.4 |
| Avaliar classificadores | Precisão, revocação, F1 | Escolha a métrica pelo custo do erro | 4.4.1 |
| Gerar texto | Modelo de linguagem (n-gramas) | Prever a próxima palavra a partir da anterior | 4.5 |
| Avaliar modelos de linguagem | Perplexidade, suavização | Medir a "surpresa" em texto novo; nunca dar probabilidade zero | 4.5.1 |
| Representar significado | *Embeddings* (PPMI + SVD, Word2Vec) | Palavras em contextos parecidos têm vetores parecidos | 4.6 |
| Significado depende do contexto | Atenção / Transformer | Cada palavra é recalculada a partir das vizinhas relevantes | 4.7 |
| Juntar tudo | LLMs | Pré-treino + ajuste por instruções + alinhamento | 4.8 |

### 🗣️ Perguntas para discussão

1. Um aluno diz: *"o ChatGPT entende português"*. Usando o que vimos (bigramas, atenção, efeito ELIZA), o que significa "entender" aqui? Onde está a fronteira entre **prever bem** e **compreender**?
2. Por que remover *stopwords* ajuda um buscador TF-IDF, mas atrapalha um analisador de sentimentos?
3. O Naive Bayes da Seção 4.4 aprendeu que "de" e "que" são negativos. Que tipo de correlação espúria um modelo de triagem de currículos poderia aprender? Como detectar isso?
4. Um modelo tem 95% de acurácia para detectar mensagens de golpe, e 95% das mensagens são legítimas. Ele é bom? Que métrica você pediria?

## 8. Vantagens e Desvantagens do PLN

### 8.1 Vantagens

**1. Automação em escala**
- Processa milhões de documentos rapidamente
- Reduz trabalho manual repetitivo

**2. Extração de insights**
- Descobre padrões em grandes volumes de texto
- Análise de tendências e sentimentos

**3. Acessibilidade**
- Interfaces conversacionais intuitivas
- Tradução automática quebra barreiras linguísticas

**4. Disponibilidade 24/7**
- Chatbots e assistentes sempre disponíveis
- Atendimento contínuo

**5. Consistência**
- Critérios uniformes de análise
- Reduz variabilidade humana

### 8.2 Desvantagens

**1. Ambiguidade linguística**
- Dificuldade com contexto, sarcasmo, ironia
- Polissemia e homonímia

**2. Viés nos dados**
- Modelos herdam preconceitos dos dados de treino
- Pode amplificar discriminação

**3. Dependência de dados**
- Modelos modernos requerem grandes volumes
- Domínios específicos com poucos dados são desafiadores

**4. Custo computacional**
- Modelos grandes (GPT, BERT) são caros para treinar
- Infraestrutura significativa necessária

**5. Falta de compreensão real**
- Sistemas não "entendem" como humanos
- Aprendizado superficial de padrões

**6. Privacidade**
- Processamento de dados sensíveis
- Preocupações com vazamento de informações

**7. Variação linguística**
- Gírias, dialetos, neologismos são desafiadores
- Linguagem evolui constantemente

## 9. Exercícios Propostos

### Exercício 1 — Tokenizador Avançado
Estenda a classe `TokenizadorSimples` para:
- Preservar URLs e emails intactos
- Tratar números com separadores (1.000,50)
- Identificar hashtags e menções (@usuario)
- Teste com tweets reais

### Exercício 2 — Classificador Naive Bayes para Texto
Implemente um classificador Naive Bayes do zero para classificação de textos:
- Use probabilidade condicional: P(classe|documento) ∝ P(classe) × ∏ P(palavra|classe)
- Treine em dataset de reviews (positivo/negativo)
- Compare com BoW vs. TF-IDF como features
- Avalie com validação cruzada

### Exercício 3 — Análise de Sentimentos com N-grams
Melhore o analisador de sentimentos usando bigramas e trigramas:
- "não gostei" deve ser tratado como uma unidade
- "muito bom" deve ter score diferente de apenas "bom"
- Implemente extração de n-grams
- Compare performance com abordagem de palavras únicas

### Exercício 4 — Sistema de Busca com TF-IDF
Crie um sistema de busca para coleção de documentos:
- Indexe documentos usando TF-IDF
- Dada uma query, retorne documentos mais relevantes (top-5)
- Use similaridade de cosseno para ranqueamento
- Implemente busca booleana (AND, OR, NOT)
- Teste com corpus de notícias

### Exercício 5 — Extrator de Palavras-Chave
Desenvolva extrator automático de palavras-chave usando múltiplas técnicas:
- TF-IDF (como baseline)
- TextRank (algoritmo baseado em grafo, similar a PageRank)
- RAKE (Rapid Automatic Keyword Extraction)
- Compare resultados das três abordagens em conjunto de documentos
- Avalie qualitativamente quais palavras-chave são mais relevantes

## 10. Referências

### Livros Fundamentais

1. **Jurafsky, D. & Martin, J. H.** (2020). *Speech and Language Processing* (3ª ed.). Disponível em: https://web.stanford.edu/~jurafsky/slp3/ — Referência completa e atualizada em PLN.

2. **Manning, C. D. & Schütze, H.** (1999). *Foundations of Statistical Natural Language Processing*. MIT Press. — Fundamentos estatísticos do PLN.

3. **Eisenstein, J.** (2019). *Introduction to Natural Language Processing*. MIT Press. — Abordagem moderna incluindo deep learning.

4. **Bird, S., Klein, E., & Loper, E.** (2009). *Natural Language Processing with Python*. O'Reilly. — Guia prático com NLTK.

5. **Goldberg, Y.** (2017). *Neural Network Methods for Natural Language Processing*. Morgan & Claypool. — Deep learning para PLN.

### Artigos Seminais

6. **Vaswani, A. et al.** (2017). "Attention Is All You Need". *NeurIPS*. — Introduz arquitetura Transformer.

7. **Mikolov, T. et al.** (2013). "Efficient Estimation of Word Representations in Vector Space". *arXiv*. — Word2Vec.

8. **Devlin, J. et al.** (2018). "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding". *arXiv*. — BERT revoluciona PLN.

9. **Brown, T. et al.** (2020). "Language Models are Few-Shot Learners". *NeurIPS*. — GPT-3.

10. **Pennington, J., Socher, R., & Manning, C.** (2014). "GloVe: Global Vectors for Word Representation". *EMNLP*.

### Recursos em Português

11. **Gonçalo Oliveira, H. et al.** (2012). "PAPEL: A Dictionary-based Lexical Ontology for Portuguese". — Recurso léxico para português.

12. **Hartmann, N. et al.** (2014). "A Large Corpus of Product Reviews in Portuguese: Tackling Out-Of-Vocabulary Words". *LREC*.

13. **Souza, F. et al.** (2020). "BERTimbau: Pretrained BERT Models for Brazilian Portuguese". *BRACIS*.

### Recursos Online

14. **NLTK Documentation**: https://www.nltk.org/ — Documentação completa com tutoriais.

15. **spaCy Documentation**: https://spacy.io/ — Guias e modelos para múltiplos idiomas.

16. **Hugging Face**: https://huggingface.co/ — Hub de modelos Transformers pré-treinados.

17. **Papers With Code - NLP**: https://paperswithcode.com/area/natural-language-processing — Estado-da-arte em PLN com código.

---

**Fim do Notebook — Processamento de Linguagem Natural**